# Section A - Data Cleaning & Analysis (Q1-154)

In [ ]:
import pandas as pd
import numpy as np
import re, datetime, warnings
from pathlib import Path
# Shared, unit-tested helpers (same folder; covered by test_preprocessing.py)
from preprocessing import (SYMBOL_TO_ISO, UNLABELLED, TARGET_CCY, RATES, RATES_SOURCE,
                           RATES_DATE, REGION_MAP, extract_currency, convert_to_target,
                           to_num, to_date, unify_flag, standardise_region)
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 200)
CANDS = [Path.cwd() / "Bank Records 1.xlsx"]
DATA = next((p for p in CANDS if p.exists()), None)
if DATA is None:
    for p in Path.cwd().rglob("Bank Records 1.xlsx"):
        DATA = p; break
print("DATA =", DATA)
OUTDIR = Path(DATA).parent
df = pd.read_excel(DATA)
raw = df.copy()
print("shape:", df.shape)
df.rename(columns={"customer_lifetime_value_amount (CLV)": "clv"}, inplace=True)

# ---------- A2: inspect python types inside object columns ----------
print("Mixed Python types in object columns:")
for c in raw.select_dtypes(include="object").columns:
    t = raw[c].dropna().map(type).value_counts()
    if len(t) > 1:
        print(" ", c, "->", {k.__name__: int(v) for k, v in t.items()})

# ---------- B1: profile currency labels BEFORE any numeric conversion ----------
MONEY = [c for c in ["account_balance", "Fixed_Deposite", "Credit_Amount"] if c in df.columns]
print("\nCurrency label profile (B1) - before conversion:")
for c in MONEY:
    print(c)
    print(raw[c].apply(extract_currency).value_counts(dropna=False).to_string())

# ---------- B5/B7: capture ISO 4217 per row, then convert to ONE target currency ----------
for c in MONEY:
    df[c + "_ccy"] = raw[c].apply(extract_currency)          # ISO 4217 per row (B5)
    df[c] = [convert_to_target(to_num(v), iso) for v, iso in zip(raw[c], df[c + "_ccy"])]
print(f"\nRates: {RATES} (source: {RATES_SOURCE}, as of {RATES_DATE}) (B7)")
print("UNLABELLED (assumed INR) rows per column:",
      {c + "_ccy": int((df[c + "_ccy"] == UNLABELLED).sum()) for c in MONEY})

# ---------- C5/C6/C8: parse dates, then validate against year/month columns ----------
df["txn_date"] = raw["txn_date"].apply(to_date)
_month_mism = int((df["txn_date"].notna() & raw["month"].notna()
                   & (df["txn_date"].dt.month != raw["month"])).sum())
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _mfirst = raw["txn_date"].apply(
        lambda v: pd.to_datetime(v, errors="coerce") if isinstance(v, str) else pd.NaT)
_amb = raw["txn_date"].astype(str).str.match(r"^\d{1,2}/\d{1,2}/\d{2,4}$")
_swaps = int((_amb & df["txn_date"].notna() & _mfirst.notna()
              & (df["txn_date"] != _mfirst)).sum())
_future = int((df["txn_date"] > pd.Timestamp.today()).sum())
print(f"\nDate validation: month-column mismatches={_month_mism}, "
      f"day/month swap candidates={_swaps}, future dates={_future}")

# ---------- D2/D6: unify flag encodings (1/0, Yes/No, '100%'/'0%') ----------
df["aml_flag"] = raw["aml_flag"].apply(unify_flag)
df["suspicious_txn_flag"] = raw["suspicious_txn_flag"].apply(unify_flag)
print("\naml_flag:", df["aml_flag"].value_counts(dropna=False).to_dict())
print("suspicious_txn_flag:", df["suspicious_txn_flag"].value_counts(dropna=False).to_dict())

# ---------- region: standardised ONCE via REGION_MAP, reused by every chart ----------
df["region_raw"] = df["region"]
df["region"] = raw["region"].apply(standardise_region)
print("\nregion (standardised):", df["region"].value_counts(dropna=False).to_dict())

print("\npreprocessing done:", df.shape)


## Assumptions & Data-Quality Issues Log (I2/I4)

### Assumptions
1. **Currency** - every money cell carries an ISO 4217 label (EUR/INR/USD, including
   the mojibake `â‚¬`/`â‚¹` sequences) or is an *unlabelled* plain number. Unlabelled
   numbers are assumed to already be in the target currency (INR) and are flagged with
   `<col>_ccy = "UNLABELLED"` so the assumption stays visible.
2. **Rates** - conversions use a static documented table:
   `RATES = {INR: 1.0, USD: 83.5, EUR: 90.4}`, RBI reference rates as of 2025-01-01.
3. **Region** - one explicit `REGION_MAP` maps the 18 raw aliases (W1/W2/W3, SNorth,
   ENorth, Upper/Lower/Middle Region, `north east`, ...) to 8 canonical regions; applied
   once and reused by every chart.
4. **Dates** - `txn_date` and the loan offer-date columns are parsed format-by-format
   (not one blanket `errors="coerce"` call) and then cross-validated against the
   provided `year`/`month` columns.
5. **Loan join** - Loan_Records is many-rows-per-customer; it is aggregated to the
   latest offer per `customer_id` before a `validate="m:1"` join, so bank row count
   is preserved.
6. **Structural nulls** - `payment_gateway`/`txn_id_external` are only expected for
   certain txn_types; missing values there are structural, not true data loss.

## Key Findings (I3)

- Money columns mix four encodings (mojibake-euro, mojibake-rupee, `INR`/`USD` text,
  and plain numbers); after converting to INR the average account balance is computed
  on a consistent basis instead of silently summing mixed currencies.
- `aml_flag` arrived as `1/0` **and** `'Yes'/'No'`; `suspicious_txn_flag` arrived as
  `'100%'/'0%'` text. Both are unified to 1/0 in the preprocessing cell above.
- The `region` column has 18 raw spellings for what should be 8 regions; the single
  `REGION_MAP` removes that double counting from every regional chart.
- `txn_date` is stored in four formats; cross-checking parsed months against the
  `month` column surfaces hidden day/month swaps instead of trusting one parser.
- Loan_Records has ~500 duplicate customers; an uncontrolled join would inflate the
  bank table from 13,002 to >14,000 rows. The `validate="m:1"` join keeps it at 13,002.

## Assumptions & Data-Quality Issues Log (I2/I4)

### Assumptions
1. **Currency** - every money cell carries an ISO 4217 label (EUR/INR/USD, including
   the mojibake `â‚¬`/`â‚¹` sequences) or is an *unlabelled* plain number. Unlabelled
   numbers are assumed to already be in the target currency (INR) and are flagged with
   `<col>_ccy = "UNLABELLED"` so the assumption stays visible.
2. **Rates** - conversions use a static documented table:
   `RATES = {INR: 1.0, USD: 83.5, EUR: 90.4}`, RBI reference rates as of 2025-01-01.
3. **Region** - one explicit `REGION_MAP` maps the 18 raw aliases (W1/W2/W3, SNorth,
   ENorth, Upper/Lower/Middle Region, `north east`, ...) to 8 canonical regions; applied
   once and reused by every chart.
4. **Dates** - `txn_date` and the loan offer-date columns are parsed format-by-format
   (not one blanket `errors="coerce"` call) and then cross-validated against the
   provided `year`/`month` columns.
5. **Loan join** - Loan_Records is many-rows-per-customer; it is aggregated to the
   latest offer per `customer_id` before a `validate="m:1"` join, so bank row count
   is preserved.
6. **Structural nulls** - `payment_gateway`/`txn_id_external` are only expected for
   certain txn_types; missing values there are structural, not true data loss.

## Key Findings (I3)

- Money columns mix four encodings (mojibake-euro, mojibake-rupee, `INR`/`USD` text,
  and plain numbers); after converting to INR the average account balance is computed
  on a consistent basis instead of silently summing mixed currencies.
- `aml_flag` arrived as `1/0` **and** `'Yes'/'No'`; `suspicious_txn_flag` arrived as
  `'100%'/'0%'` text. Both are unified to 1/0 in the preprocessing cell above.
- The `region` column has 18 raw spellings for what should be 8 regions; the single
  `REGION_MAP` removes that double counting from every regional chart.
- `txn_date` is stored in four formats; cross-checking parsed months against the
  `month` column surfaces hidden day/month swaps instead of trusting one parser.
- Loan_Records has ~500 duplicate customers; an uncontrolled join would inflate the
  bank table from 13,002 to >14,000 rows. The `validate="m:1"` join keeps it at 13,002.

## Preprocessing tests - to_num / to_date

In [2]:
import pandas as pd, numpy as np
test_vals = pd.DataFrame({
    "raw_num": ["INR 200", "$10k", "500", None, 42],
    "raw_date": ["February4-February-2025", "28122026", "15/07/2022", None, "not-a-date"],
})
test_vals["num_out"] = test_vals["raw_num"].apply(to_num)
test_vals["date_out"] = test_vals["raw_date"].apply(to_date)
print(test_vals.to_string())
assert test_vals.loc[1, "num_out"] == 10000.0
assert test_vals.loc[4, "num_out"] == 42
assert pd.isna(test_vals.loc[3, "num_out"])
assert str(test_vals.loc[0, "date_out"].date()) == "2025-02-04"
assert str(test_vals.loc[2, "date_out"].date()) == "2022-07-15"
assert pd.isna(test_vals.loc[4, "date_out"])
print("All preprocessing tests passed")


   raw_num                 raw_date  num_out   date_out
0  INR 200  February4-February-2025    200.0 2025-02-04
1     $10k                 28122026  10000.0 2026-12-28
2      500               15/07/2022    500.0 2022-07-15
3     None                      NaN      NaN        NaT
4       42               not-a-date     42.0        NaT
All preprocessing tests passed


## Q1 - Count the total number of transactions in the dataset.

In [3]:
total_trans = len(df)
print(total_trans)

13002


## Q2 - Identify rows where customer_id is NULL.

In [4]:
null_cus = df[df["customer_id"].isnull()]
print(null_cus)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  averag_high_txn_weekend_flag averag_high_txn_day_of_week txn_velocity_1hr device_change_flag  \
11             NaN      5062.0  TXN-4857-5062-17-TXN674500        17.0            34%  2025.0  ...                           1.0                         Sun              2.0                 No   
19             NaN      3084.0  TXN-2655-3084-15-TXN615232        15.0            43%  2023.0  ...                           1.0                         Sat              0.0                 No   
35             NaN      2557.0  TXN-3779-2557-04-TXN964882         4.0            94%  2025.0  ...                           1.0                         Sun              2.0                 No   
69             NaN      5568.0  TXN-0560-5568-09-TXN575543         9.0            31%  2026.0  ...                           1.0                         Sun              0.0                 No   
93             NaN  

## Q3 - The dataset has missing kyc_status values. Find all transactions where kyc_status has not been completed yet.

In [5]:
kyc_incomplete = df[df["kyc_status"] == "Pending"]
print(kyc_incomplete)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  averag_high_txn_weekend_flag averag_high_txn_day_of_week txn_velocity_1hr device_change_flag  \
5           1820.0      3089.0  TXN-1820-3089-13-TXN983778        13.0            60%  2026.0  ...                           1.0                         Sun              1.0                 No   
8             86.0      2849.0  TXN-0086-2849-07-TXN990616         7.0            32%  2026.0  ...                           1.0                         Sat              1.0                 No   
9           4347.0      6968.0  TXN-4347-6968-12-TXN655229        12.0            19%  2023.0  ...                           1.0                         Sun              2.0                 No   
10           898.0      5120.0  TXN-0898-5120-21-TXN325000        21.0            NaN  2026.0  ...                           1.0                         Sat              2.0                 No   
16           500.0  

## Q4 - Build a filtered list containing only valid payment transactions where payment_gateway is available.

In [6]:
valid_pay = df[df["payment_gateway"].notnull()]
print(valid_pay)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  averag_high_txn_weekend_flag averag_high_txn_day_of_week txn_velocity_1hr device_change_flag  \
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...                           1.0                         Sat              4.0                 No   
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...                           1.0                         Sat              0.0                 No   
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...                           1.0                         Sun              1.0                 No   
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...                           1.0                         Sun              1.0                 No   
4           2990.0  

## Q5 - Extract numeric values from the account_balance column which contains currency symbols and text.

In [7]:
df["account_balance"] = (df["account_balance"].astype(str).str.extract(r"(-?\d+(?:\.\d+)?)")[0].astype(float))
print(df["account_balance"])

0         8216.04
1        -1354.76
2          872.32
3          609.70
4         1799.68
           ...   
12997    10000.00
12998     1254.34
12999     1640.55
13000    10000.00
13001     2004.92
Name: account_balance, Length: 13002, dtype: float64


## Q6 - Clean currency-formatted values (€, $, ₹, text) across account_balance, Fixed_Deposite, and Credit_Amount into pure numeric format.

In [8]:
columns = ["account_balance", "Fixed_Deposite", "Credit_Amount"]
for column in columns:
    df[column] = (
        df[column]
        .astype(str).str.replace(",", "", regex=False).str.extract(r"(-?\d+(?:\.\d+)?)")[0].astype(float)
    )
print(df[columns])

       account_balance  Fixed_Deposite  Credit_Amount
0              8216.04         8561.29        2406.56
1             -1354.76         2811.49          68.78
2               872.32         6191.72         554.87
3               609.70         1875.70        1258.98
4              1799.68         3804.99       10000.00
...                ...             ...            ...
12997         10000.00             NaN        3833.98
12998          1254.34        -3453.19         -59.64
12999          1640.55         5391.61        1940.12
13000         10000.00          745.73       10000.00
13001          2004.92        -1065.37         298.86

[13002 rows x 3 columns]


## Q7 - Remove leading and trailing spaces from the payment_method column.

In [9]:
df["payment_method"] = df["payment_method"].str.strip()
print(df["payment_method"])

0                UPI
1        Net Banking
2                UPI
3             Wallet
4        Credit Card
            ...     
12997         Wallet
12998         Wallet
12999         Wallet
13000    Credit Card
13001            UPI
Name: payment_method, Length: 13002, dtype: str


## Q8 - Identify duplicate transactions by checking both transaction_id and amount columns together.

In [10]:
duplicate_transactions = df[df.duplicated(subset=["transaction_id","account_balance"])]
print(duplicate_transactions)

       customer_id  account_id transaction_id  product_id product_margin    year  ...  averag_high_txn_weekend_flag averag_high_txn_day_of_week txn_velocity_1hr device_change_flag  \
481         2774.0         NaN            NaN         7.0            31%  2021.0  ...                           1.0                         Sun              0.0                 No   
1631        3569.0      6319.0            NaN        15.0            18%  2026.0  ...                           1.0                         Sat              2.0                 No   
1731        2834.0      3968.0            NaN        20.0            26%  2026.0  ...                           1.0                         Sun              0.0                 No   
2140        4750.0      5938.0            NaN         2.0            11%  2026.0  ...                           1.0                         Sun              2.0                 No   
2217         821.0      4628.0            NaN         3.0            92%  2025.0  ...

## Q9 - Count how many records have NULL in the payment_method column.

In [11]:
nullPayMet = df["payment_method"].isnull().sum()
print(nullPayMet)

244


## Q10 - Standardize region values to fix case inconsistencies like 'south east' to 'South-East'.

In [ ]:
# Q10 reuses the single REGION_MAP applied once during preprocessing, so the
# standardised region is consistent in every downstream chart (review fix).
df["region"] = raw["region"].apply(standardise_region)
print(df["region"].value_counts(dropna=False))


## Q11 - Differentiate between actual 0 values (integer zero) and NULL/NaN values across all columns.

In [13]:
for c in df.columns:
    zerocnt = (df[c] == 0).sum()
    nullcnt = df[c].isnull().sum()
print(c, "zero: ", zerocnt, "NULL: ", nullcnt)

failed_txn_count_24hr zero:  11452 NULL:  257


## Q12 - Identify rows where txn_date contains invalid or unparseable date formats.

In [14]:
invalid_date = df[
    pd.to_datetime(df["txn_date"], errors="coerce").isnull()
    & df["txn_date"].notnull()
]
print(invalid_date)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr]
Index: []

[0 rows x 41 columns]


## Q13 - Standardize all txn_date values into a single YYYY-MM-DD datetime format.

In [15]:
df["txn_date"] = pd.to_datetime(df["txn_date"], errors="coerce")
df["txn_date"] = df["txn_date"].dt.strftime("%Y-%m-%d")
print(df["txn_date"])

0        2024-10-01
1        2022-07-15
2        2024-02-18
3        2025-02-04
4        2024-04-17
            ...    
12997    2023-03-14
12998    2022-09-15
12999    2022-04-19
13000    2024-03-16
13001    2021-04-28
Name: txn_date, Length: 13002, dtype: str


## Q14 - Categorize transactions with negative account_balance values as either 'Refund' or 'Error'.

In [16]:
df["balance_category"] = "Normal"
df.loc[(df["account_balance"] < 0) & (df["txn_type"] == "Refund"), "balance_category"] = "Refund"
df.loc[(df["account_balance"] < 0) & (df["txn_type"] != "Refund"), "balance_category"] = "Error"
print(df[["account_balance", "txn_type", "balance_category"]])

       account_balance    txn_type balance_category
0              8216.04  Withdrawal           Normal
1             -1354.76      Refund           Refund
2               872.32      Refund           Normal
3               609.70     Deposit           Normal
4              1799.68    Purchase           Normal
...                ...         ...              ...
12997         10000.00     Deposit           Normal
12998          1254.34  Withdrawal           Normal
12999          1640.55     Deposit           Normal
13000         10000.00      Refund           Normal
13001          2004.92      Refund           Normal

[13002 rows x 3 columns]


## Q15 - Replace empty strings ('') with NaN values across the entire dataset.

In [17]:
print(df.replace("", np.nan, inplace=True))

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  averag_high_txn_day_of_week txn_velocity_1hr device_change_flag location_change_flag  \
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...                          Sat              4.0                 No                   No   
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...                          Sat              0.0                 No                   No   
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...                          Sun              1.0                 No                   No   
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...                          Sun              1.0                 No                   No   
4           2990.0      4126.0  TXN-2990-4126-15-TXN890459  

## Q16 - Standardize casing in the payment_status column.

In [18]:
df["payment_status"] = df["payment_status"].str.strip().str.title()
print("rows cleaned:", len(df))
print(df["payment_status"].value_counts(dropna=False).to_string())


rows cleaned: 13002
payment_status
Refunded    3263
Failed      3194
Success     3185
Pending     3127
NaN          233


## Q17 - Detect conflicting transactions where the same transaction_id has different amounts.

In [19]:
conflict = df.groupby("transaction_id")["account_balance"].nunique()
conflict = conflict[conflict > 1]
print(conflict)

Series([], Name: account_balance, dtype: int64)


## Q18 - Find transactions where payment_gateway is NULL but payment_status is 'Success'.

In [20]:
gat = df[df["payment_gateway"].isnull() & (df["payment_status"] == "Success")]
print(gat)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  averag_high_txn_day_of_week txn_velocity_1hr device_change_flag location_change_flag  \
130         4171.0      3229.0  TXN-4171-3229-17-TXN155552        17.0            23%  2022.0  ...                          Sun              0.0                 No                   No   
165          199.0       626.0  TXN-0199-0626-06-TXN104788         6.0            47%  2021.0  ...                          Sat              1.0                 No                   No   
182         4095.0      3759.0  TXN-4095-3759-15-TXN313172        15.0            67%  2021.0  ...                          Sun              1.0                 No                   No   
343         2867.0      5158.0  TXN-2867-5158-16-TXN619152        16.0            75%  2025.0  ...                          Sat              2.0                 No                   No   
395         2246.0      7863.0  TXN-2246-7863-05-TXN502456  

## Q19 - Standardize inconsistent values in the txn_mode column.

In [21]:
df["txn_mode"] = df["txn_mode"].str.title()
print(df["txn_mode"].unique())

<StringArray>
['Online', 'Offline', nan]
Length: 3, dtype: str


## Q20 - Count transactions where product_id is NULL.

In [22]:
nullId = df["product_id"].isnull().sum()
print(nullId)

230


## Q21 - Identify transactions where amount = 0 and payment_status = 'Success'.

In [23]:
tran = df[(df["account_balance"] == 0) & (df["payment_status"] == "Success")]
print(tran)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  averag_high_txn_day_of_week txn_velocity_1hr device_change_flag location_change_flag  \
10847       3619.0         NaN  TXN-3619-0000-01-TXN986365         1.0            38%  2023.0  ...                          Sat              0.0                 No                   No   
11229       1795.0         NaN  TXN-1795-0000-01-TXN758765         1.0            86%  2026.0  ...                          Sun              1.0                 No                  Yes   
11683       3607.0         NaN  TXN-3607-0000-01-TXN178463         1.0            52%  2021.0  ...                          Sat              2.0                 No                   No   

      failed_txn_count_24hr balance_category  
10847                   0.0           Normal  
11229                   0.0           Normal  
11683                   0.0           Normal  

[3 rows x 42 columns]


## Q22 - Clean and standardize inconsistent values in the txn_type column.

In [24]:
df["txn_type"] = df["txn_type"].str.strip().str.title()
print(df["txn_type"].unique())

<StringArray>
['Withdrawal', 'Refund', 'Deposit', 'Purchase', nan]
Length: 5, dtype: str


## Q23 - Standardize payment_method values (e.g., map 'CC' to 'Credit Card', 'DC' to 'Debit Card').

In [25]:
df["payment_method"] = df["payment_method"].replace({
    "CC": "Credit Card",
    "DC": "Debit Card",
    "Dc" : "Debit Card",
    "Cc" : "Credit Card"
})
print(df["payment_method"].unique())

<StringArray>
['UPI', 'Net Banking', 'Wallet', 'Credit Card', 'Debit Card', nan]
Length: 6, dtype: str


## Q24 - Identify duplicate customer_id and txn_date combinations.

In [26]:
dupli = df[df.duplicated(subset=["customer_id", "txn_date"], keep=False)]
print(dupli)

       customer_id  account_id               transaction_id  product_id product_margin    year  ...  averag_high_txn_day_of_week txn_velocity_1hr device_change_flag location_change_flag  \
5           1820.0      3089.0   TXN-1820-3089-13-TXN983778        13.0            60%  2026.0  ...                          Sun              1.0                 No                   No   
23          4644.0      6335.0   TXN-4644-6335-25-TXN724774        25.0            NaN  2026.0  ...                          Sat              0.0                 No                   No   
32          2522.0      3129.0   TXN-2522-3129-06-TXN293343         6.0             3%  2023.0  ...                          Fri              0.0                 No                   No   
39            59.0      5053.0   TXN-0059-5053-08-TXN707589         8.0             3%  2026.0  ...                          Sat              1.0                 No                   No   
54           143.0      7041.0   TXN-0143-7041-17-TXN56

## Q25 - Count transactions grouped by payment_method and payment_status.

In [27]:
result = df.groupby(["payment_method", "payment_status"]).size()
print(result)

payment_method  payment_status
Credit Card     Failed            638
                Pending           604
                Refunded          627
                Success           647
Debit Card      Failed            617
                                 ... 
UPI             Success           618
Wallet          Failed            628
                Pending           643
                Refunded          643
                Success           612
Length: 20, dtype: int64


## Q26 - Find the most frequently used payment_gateway.

In [28]:
print(df["payment_gateway"].mode()[0])

Razorpay


## Q27 - Calculate the percentage of failed transactions.

In [29]:
failcnt= (df["payment_status"] == "Failed").sum()
totalcnt = len(df)
failper = (failcnt / totalcnt) * 100
print(failper)

24.565451469004767


## Q28 - Group transactions by txn_type and count them.

In [30]:
txn_type = df.groupby("txn_type").size()
print(txn_type)

txn_type
Deposit       3150
Purchase      3172
Refund        3201
Withdrawal    3224
dtype: int64


## Q29 - Compare total number of online vs offline transactions.

In [31]:
cnt = df["txn_mode"].str.strip().str.lower().value_counts()
print(cnt)

txn_mode
online     11495
offline     1286
Name: count, dtype: int64


## Q30 - Calculate total reward_points_earned by each customer.

In [32]:
cust_reward = df.groupby("customer_id")["reward_points_earned"].sum()
print(cust_reward)

customer_id
1.0       26150.0
4.0        2780.0
5.0        4761.0
6.0        2269.0
7.0        6172.0
           ...   
4995.0      874.0
4996.0     1967.0
4998.0      483.0
4999.0     5412.0
5000.0     3234.0
Name: reward_points_earned, Length: 4318, dtype: float64


## Q31 - Identify transactions where an offer was applied.

In [33]:
offer = df[df["offer_applied_flag"] == 1]
print(offer)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  averag_high_txn_day_of_week txn_velocity_1hr device_change_flag location_change_flag  \
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...                          Sat              0.0                 No                   No   
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...                          Sun              1.0                 No                   No   
4           2990.0      4126.0  TXN-2990-4126-15-TXN890459        15.0             9%  2024.0  ...                          Sat             42.0                 No                  Yes   
5           1820.0      3089.0  TXN-1820-3089-13-TXN983778        13.0            60%  2026.0  ...                          Sun              1.0                 No                   No   
6           2762.0      4749.0  TXN-2762-4749-07-TXN585525  

## Q32 - Calculate the percentage of transactions where offers were applied.

In [34]:
offercnt = len(df[df["offer_applied_flag"] == 1])
offerper = (offercnt / len(df)) * 100
print(offerper)

49.45393016459006


## Q33 - Group transactions by customer_segment.

In [35]:
seg = df.groupby("customer_segment").size()
print(seg)

customer_segment
Corporate    3227
Gold         3208
Platinum     3216
Retail       3121
dtype: int64


## Q34 - Identify the region with the highest number of transactions.

In [36]:
region = df["region"].value_counts()
print(region.head(1))

region
North-East    1388
Name: count, dtype: int64


## Q35 - Find customers who have only one transaction.

In [37]:
trans_cnt = df["customer_id"].value_counts()
singletrans = trans_cnt[trans_cnt == 1]
print("customers with exactly 1 transaction:", len(singletrans))
print(singletrans.head(10).to_string())


customers with exactly 1 transaction: 754
customer_id
3461.0    1
2636.0    1
4377.0    1
3749.0    1
1651.0    1
3037.0    1
823.0     1
2755.0    1
2516.0    1
1394.0    1


## Q36 - Identify customers who have never used any offer.

In [38]:
nooffer_cust = df.groupby("customer_id")["offer_applied_flag"].sum()
nooffer_cust = nooffer_cust[nooffer_cust == 0]
print(nooffer_cust)

customer_id
4.0       0.0
6.0       0.0
8.0       0.0
12.0      0.0
14.0      0.0
         ... 
4990.0    0.0
4992.0    0.0
4995.0    0.0
4996.0    0.0
4998.0    0.0
Name: offer_applied_flag, Length: 914, dtype: float64


## Q37 - Calculate failure rate for each payment_gateway.

In [39]:
failrate = df[df["payment_status"] == "Failed"]
failrate = failrate.groupby("payment_gateway").size()
print(failrate)

payment_gateway
PayU        788
PhonePe     794
Razorpay    775
Stripe      775
dtype: int64


## Q38 - Calculate percentage contribution of each payment_method.

In [40]:
payment_cnt = df["payment_method"].value_counts()
payper = (payment_cnt / len(df)) * 100
print(payper)

payment_method
Wallet         19.812337
Credit Card    19.773881
Debit Card     19.658514
Net Banking    19.450854
UPI            19.427780
Name: count, dtype: float64


## Q39 - Count unique payment methods used per customer.

In [41]:
uniq = df.groupby("customer_id")["payment_method"].nunique()
print(uniq)

customer_id
1.0       5
4.0       1
5.0       3
6.0       2
7.0       2
         ..
4995.0    1
4996.0    1
4998.0    2
4999.0    4
5000.0    4
Name: payment_method, Length: 4318, dtype: int64


## Q40 - Rank customers based on total number of transactions.

In [42]:
customer = df["customer_id"].value_counts()
custrnk = customer.rank(
    method="dense",
    ascending=False
)
result = pd.DataFrame({
    "transaction_count": customer,
    "rank": custrnk
})
print(result)

             transaction_count  rank
customer_id                         
1.0                         25   1.0
2163.0                       8   2.0
3516.0                       7   3.0
578.0                        7   3.0
2263.0                       7   3.0
...                        ...   ...
925.0                        1   9.0
1899.0                       1   9.0
2199.0                       1   9.0
4655.0                       1   9.0
3593.0                       1   9.0

[4318 rows x 2 columns]


## Q41 - Rank months based on number of transactions after cleaning dates.

In [43]:
df["txn_date"] = pd.to_datetime(df["txn_date"], errors="coerce")
monthcnt = df["txn_date"].dt.month.value_counts()
monthrnk = monthcnt.rank(method="dense", ascending=False)
result = pd.DataFrame({
    "transaction_count": monthcnt,
    "rank": monthrnk
})
print(result)

          transaction_count  rank
txn_date                         
2.0                    1107   1.0
4.0                    1085   2.0
10.0                   1083   3.0
9.0                    1082   4.0
1.0                    1064   5.0
...                     ...   ...
6.0                    1040   7.0
8.0                    1040   7.0
5.0                    1038   8.0
12.0                   1012   9.0
11.0                    978  10.0

[12 rows x 2 columns]


## Q42 - Calculate ratio of successful vs failed transactions.

In [44]:
success = (df["payment_status"] == "Success").sum()
failed = (df["payment_status"] == "Failed").sum()
ratio = success / failed
print("Successful transactions:", success)
print("Failed transactions:", failed)
print("Success to Failure ratio:", round(ratio, 2))

Successful transactions: 3185
Failed transactions: 3194
Success to Failure ratio: 1.0


## Q43 - Identify customers using multiple devices or channels.

In [45]:
device = df.groupby("customer_id")["device_type"].nunique()
muldev = device[device > 1]
print(muldev)

customer_id
1.0       3
4.0       2
7.0       2
8.0       2
9.0       2
         ..
4992.0    2
4993.0    3
4994.0    3
4999.0    2
5000.0    2
Name: device_type, Length: 2891, dtype: int64


## Q44 - Count transactions grouped by day of week or month after date cleaning.

In [46]:
day = df["txn_date"].dt.day_name().value_counts()
print(day)
month = df["txn_date"].dt.month_name().value_counts()
print(month)

txn_date
Sunday       1840
Tuesday      1838
Monday       1820
Friday       1818
Wednesday    1794
Saturday     1789
Thursday     1744
Name: count, dtype: int64
txn_date
February     1107
April        1085
October      1083
September    1082
January      1064
             ... 
June         1040
August       1040
May          1038
December     1012
November      978
Name: count, Length: 12, dtype: int64


## Q45 - Identify customers transacting across multiple regions.

In [47]:
region= df.groupby("customer_id")["region"].nunique()
mulregion = region[region > 1]
print(mulregion)

customer_id
1.0       12
4.0        2
5.0        3
6.0        2
7.0        5
          ..
4993.0     4
4994.0     5
4998.0     2
4999.0     4
5000.0     5
Name: region, Length: 3390, dtype: int64


## Q46 - Categorize transactions into Low, Medium, and High value groups.

In [48]:
df["value_group"] = pd.cut(
    df["account_balance"],
    bins=[-float("inf"), 10000, 50000, float("inf")],
    labels=["Low", "Medium", "High"]
)
print(df[["account_balance", "value_group"]])

       account_balance value_group
0              8216.04         Low
1             -1354.76         Low
2               872.32         Low
3               609.70         Low
4              1799.68         Low
...                ...         ...
12997         10000.00         Low
12998          1254.34         Low
12999          1640.55         Low
13000         10000.00         Low
13001          2004.92         Low

[13002 rows x 2 columns]


## Q47 - Analyze distribution of transactions across payment gateways.

In [49]:
gateway = df["payment_gateway"].value_counts()
gatewayper = (
    df["payment_gateway"].value_counts(normalize=True) * 100
)
print(gatewayper.round(2))

payment_gateway
Razorpay    25.45
Stripe      25.09
PhonePe     24.88
PayU        24.59
Name: proportion, dtype: float64


## Q48 - Compare transaction counts across different customer segments.

In [50]:
segmentcnt = df["customer_segment"].value_counts()
segment_count = segmentcnt.sort_values(ascending=False)
print(segment_count)

customer_segment
Corporate    3227
Platinum     3216
Gold         3208
Retail       3121
Name: count, dtype: int64


## Q49 - Identify NULL handling issues across the entire dataset and rank columns by null count.

In [51]:
nullcnt = df.isnull().sum()
nullrnk = nullcnt.sort_values(ascending=False)
print(nullrnk)

offer_id                6606
txn_date                 359
Fixed_Deposite           307
account_id               302
region                   267
                        ... 
customer_id              227
location_change_flag     224
txn_mode                 221
aml_flag                 213
balance_category           0
Length: 43, dtype: int64


## Q50 - Detect incorrect grouping caused by inconsistent data values.

In [52]:
for col in ["payment_method", "txn_type", "txn_mode", "region"]:
    print("\n", col)
    print(df[col].value_counts())


 payment_method
payment_method
Wallet         2576
Credit Card    2571
Debit Card     2556
Net Banking    2529
UPI            2526
Name: count, dtype: int64

 txn_type
txn_type
Withdrawal    3224
Refund        3201
Purchase      3172
Deposit       3150
Name: count, dtype: int64

 txn_mode
txn_mode
Online     11495
Offline     1286
Name: count, dtype: int64

 region
region
North-East         1388
South-East         1355
middle region       757
east                745
us - north west     726
                   ... 
w1                  701
enorth              700
west                696
south-west          693
north               680
Name: count, Length: 16, dtype: int64


## Q51 - Validate transaction counts before and after each cleaning operation.

In [53]:
before = len(df)
df["payment_method"] = df["payment_method"].str.strip()
after = len(df)
print("Before Cleaning:", before)
print("After Cleaning:", after)
print("Difference :", before - after)

Before Cleaning: 13002
After Cleaning: 13002
Difference : 0


## Q52 - Create a data quality score per column combining null percentage, duplicate percentage, and format consistency.

In [54]:
for col in df.columns:
    nullper = (df[col].isnull().sum() / len(df)) * 100
    dupper = (df[col].duplicated().sum() / len(df)) * 100
    score = 100 - nullper - dupper
    print(col, "Data Quality Score:", round(score, 2))

customer_id Data Quality Score: 31.47
account_id Data Quality Score: 45.56
transaction_id Data Quality Score: 96.36
product_id Data Quality Score: -1.57
product_margin Data Quality Score: -1.22
year Data Quality Score: -1.92
month Data Quality Score: -1.77
region Data Quality Score: -1.92
customer_segment Data Quality Score: -1.73
kyc_status Data Quality Score: -1.78
txn_id_external Data Quality Score: 96.21
txn_type Data Quality Score: -1.92
txn_mode Data Quality Score: -1.68
payment_method Data Quality Score: -1.83
payment_gateway Data Quality Score: -1.77
payment_status Data Quality Score: -1.75
device_type Data Quality Score: -1.85
app_version Data Quality Score: -1.79
offer_id Data Quality Score: -50.65
offer_category Data Quality Score: -1.87
offer_applied_flag Data Quality Score: -1.8
reward_points_earned Data Quality Score: 13.52
offer_effectiveness Data Quality Score: -1.24
account_balance Data Quality Score: 69.1
Fixed_Deposite Data Quality Score: 69.18
Credit_Amount Data Qua

## Q53 - Perform a full audit of the dataset – generate a summary showing column name, data type, null count, unique count, and sample values.

In [55]:
audit = pd.DataFrame({
    "column_name": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "null_count": df.isnull().sum().values,
    "unique_count": df.nunique().values,
    "sample_value": df.iloc[0].values
})
print(audit)

              column_name data_type  null_count  unique_count                sample_value
0             customer_id   float64         227          4318                      4026.0
1              account_id   float64         302          6225                      5410.0
2          transaction_id       str         237         12765  TXN-4026-5410-01-TXN278384
3              product_id   float64         230            25                         1.0
4          product_margin       str         260           101                         91%
..                    ...       ...         ...           ...                         ...
38     device_change_flag       str         234             2                          No
39   location_change_flag       str         224             2                          No
40  failed_txn_count_24hr   float64         257            18                         0.0
41       balance_category       str           0             3                      Normal
42        

## Q54 - Calculate total transaction amount per customer along with their customer_id and region.

In [56]:
res = df.groupby(
    ["customer_id", "region"]
)["Credit_Amount"].sum().reset_index()
res = res.rename(
    columns={"Credit_Amount": "total_transaction_amount"}
)
print(res)

       customer_id         region  total_transaction_amount
0              1.0     South-East                  10060.12
1              1.0           east                   -142.69
2              1.0         enorth                  10000.00
3              1.0   lower region                    277.97
4              1.0  middle region                   3567.19
...            ...            ...                       ...
10893       5000.0          north                  -1483.85
10894       5000.0         snorth                   2363.77
10895       5000.0          south                  20000.00
10896       5000.0   upper region                  10000.00
10897       5000.0             w3                  10000.00

[10898 rows x 3 columns]


## Q55 - Filter only successful transactions before computing any aggregations.

In [57]:
successTxn = df[df["payment_status"] == "Success"]
result = successTxn.groupby("customer_id")["account_balance"].sum()
print(result)

customer_id
1.0       999299.00
4.0          818.76
5.0        10000.00
7.0        15884.69
10.0       10000.00
            ...    
4990.0      1041.55
4992.0      7141.62
4993.0     12093.19
4994.0      1913.23
5000.0     10000.00
Name: account_balance, Length: 2221, dtype: float64


## Q56 - Count transactions per customer_id after filtering by a specific txn_type.

In [58]:
txn = df[df["txn_type"] == "Purchase"]
txncnt = txn.groupby("customer_id").size()
print(txncnt)

customer_id
1.0       9
6.0       2
8.0       1
9.0       1
10.0      1
         ..
4990.0    1
4991.0    1
4992.0    1
4998.0    2
5000.0    1
Length: 2211, dtype: int64


## Q57 - Compute per-customer summary statistics and combine them back with the original transaction data.

In [59]:
sum = df.groupby("customer_id")["account_balance"].agg(["sum", "mean", "count"])
resu = df.merge(sum, on="customer_id")
print(resu)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  failed_txn_count_24hr balance_category value_group       sum         mean count
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...                    0.0           Normal         Low   8216.04  8216.040000     1
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...                    0.0           Refund         Low   4684.29  2342.145000     2
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...                    0.0           Normal         Low   9937.94  1987.588000     5
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...                    0.0           Normal         Low   2125.26   531.315000     4
4           2990.0      4126.0  TXN-2990-4126-15-TXN890459        15.0             9%  2024.0  ...  

## Q58 - Join the transaction data with a customer-level aggregated summary on customer_id.

In [60]:
summary = df.groupby("customer_id")["account_balance"].sum().reset_index()
summary = summary.rename(
    columns={"Credit Amount": "total_amount"}
)
r = df.merge(summary, on="customer_id", how="left")
print(r)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  device_change_flag location_change_flag failed_txn_count_24hr balance_category value_group  \
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...                  No                   No                   0.0           Normal         Low   
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...                  No                   No                   0.0           Refund         Low   
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...                  No                   No                   0.0           Normal         Low   
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...                  No                   No                   0.0           Normal         Low   
4           2990.0      4126.0

## Q59 - Select customer_id along with aggregated values like sum, mean, and count while handling non-aggregated columns.

In [61]:
result = df.groupby("customer_id").agg({
"account_balance": ["sum", "mean", "count"]
})
print(result)

            account_balance                    
                        sum          mean count
customer_id                                    
1.0              1017960.61  40718.424400    25
4.0                10818.76   5409.380000     2
5.0                11541.99   5770.995000     2
6.0                11281.34   5640.670000     2
7.0                21136.36   3522.726667     6
...                     ...           ...   ...
4995.0             -1667.50  -1667.500000     1
4996.0              4957.98   4957.980000     1
4998.0             21987.75  10993.875000     2
4999.0             13824.62   2764.924000     5
5000.0             30010.18   5001.696667     6

[4318 rows x 3 columns]


## Q60 - First filter raw transaction rows by a condition, then group them and compute aggregated results.

In [62]:
filter = df[df["payment_status"] == "Success"]
result = filter.groupby("region")["account_balance"].sum()
print(result)

region
North-East         1349108.07
South-East         1276035.76
east                786809.34
enorth              791305.80
lower region        705289.49
                      ...    
us - north west     589613.50
w1                  486773.05
w2                  657313.43
w3                  594646.66
west               1680037.51
Name: account_balance, Length: 16, dtype: float64


## Q61 - Compare results of size() vs count() vs sum() on customer_id to analyze NULL impact on counting.

In [63]:
print("size():")
print(df.groupby("customer_id").size())
print("count():")
print(df.groupby("customer_id")["customer_id"].count())
print("sum() on reward points:")
print(df.groupby("customer_id")["reward_points_earned"].sum())

size():
customer_id
1.0       25
4.0        2
5.0        3
6.0        2
7.0        6
          ..
4995.0     1
4996.0     1
4998.0     2
4999.0     6
5000.0     6
Length: 4318, dtype: int64
count():
customer_id
1.0       25
4.0        2
5.0        3
6.0        2
7.0        6
          ..
4995.0     1
4996.0     1
4998.0     2
4999.0     6
5000.0     6
Name: customer_id, Length: 4318, dtype: int64
sum() on reward points:
customer_id
1.0       26150.0
4.0        2780.0
5.0        4761.0
6.0        2269.0
7.0        6172.0
           ...   
4995.0      874.0
4996.0     1967.0
4998.0      483.0
4999.0     5412.0
5000.0     3234.0
Name: reward_points_earned, Length: 4318, dtype: float64


## Q62 - Retrieve the first 100 transactions.

In [64]:
first100 = df.head(100)
print(first100)

    customer_id  account_id              transaction_id  product_id product_margin    year  ...  txn_velocity_1hr device_change_flag location_change_flag failed_txn_count_24hr balance_category  \
0        4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...               4.0                 No                   No                   0.0           Normal   
1        2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...               0.0                 No                   No                   0.0           Refund   
2        3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...               1.0                 No                   No                   0.0           Normal   
3         408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...               1.0                 No                   No                   0.0           Normal   
4        2990.0     

## Q63 - Implement pagination to fetch transactions in pages (e.g., first 100, next 100).

In [65]:
page1 = df.iloc[0:100]
page2 = df.iloc[100:200]
print("Page 1")
print(page1)
print("Page 2")
print(page2)

Page 1
    customer_id  account_id              transaction_id  product_id product_margin    year  ...  txn_velocity_1hr device_change_flag location_change_flag failed_txn_count_24hr balance_category  \
0        4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...               4.0                 No                   No                   0.0           Normal   
1        2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...               0.0                 No                   No                   0.0           Refund   
2        3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...               1.0                 No                   No                   0.0           Normal   
3         408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...               1.0                 No                   No                   0.0           Normal   
4        2990

## Q64 - Access column values directly without quoting column names as strings.

In [66]:
cust_ids = df.customer_id
print(cust_ids)

0        4026.0
1        2249.0
2        3542.0
3         408.0
4        2990.0
          ...  
12997    1635.0
12998    4164.0
12999      18.0
13000    3821.0
13001    4761.0
Name: customer_id, Length: 13002, dtype: float64


## Q65 - Join the transaction dataset with itself on customer_id while avoiding Cartesian product.

In [67]:
x = df[["customer_id", "transaction_id"]].copy()
result = df.merge(
    x,
    on="customer_id",
    suffixes=("_transaction1", "_transaction2")
)
result = result[
    result["transaction_id_transaction1"] !=
    result["transaction_id_transaction2"]
]
print(result)

       customer_id  account_id transaction_id_transaction1  product_id product_margin    year  ...  device_change_flag location_change_flag failed_txn_count_24hr balance_category value_group  \
1           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...                  No                   No                   0.0           Normal         Low   
3           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...                  No                   No                   0.0           Refund         Low   
5           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...                  No                   No                   0.0           Normal         Low   
6           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...                  No                   No                   0.0           Normal         Low   
7           3542.0       133.0

## Q66 - Identify duplicate rows and remove them without relying on DISTINCT-like deduplication.

In [68]:
duplicate = df[df.duplicated()]
print(duplicate)
no_duplicate = df.drop_duplicates()
print(no_duplicate)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr, balance_category, value_group]
Index: []

[0 rows x 43 columns]
       customer_id  account_id              transaction_id  product_id product_margin    year  ...  txn_velocity_1hr device_change_flag location_change_flag failed_txn_count_24hr balance_category  \
0           4026.0      5410.0  TXN-402

## Q67 - Retrieve a limited number of rows (like LIMIT) and skip rows (like OFFSET).

In [69]:
limitrows = df.iloc[50:100]
print(limitrows)

    customer_id  account_id              transaction_id  product_id product_margin    year  ...  txn_velocity_1hr device_change_flag location_change_flag failed_txn_count_24hr balance_category  \
50        489.0      1406.0  TXN-0489-1406-14-TXN999462        14.0            65%  2024.0  ...               4.0                 No                   No                   0.0           Normal   
51       4165.0      1406.0  TXN-4165-1406-03-TXN645602         3.0            12%  2026.0  ...               1.0                 No                   No                   1.0           Normal   
52        391.0      3262.0  TXN-0391-3262-15-TXN686727        15.0             0%  2021.0  ...               3.0                 No                  Yes                   0.0           Normal   
53        812.0      4778.0  TXN-0812-4778-05-TXN437803         5.0            20%  2026.0  ...               2.0                 No                   No                   1.0           Normal   
54        143.0     

## Q68 - Filter transactions where payment_status is 'Success' after handling case variations.

In [70]:
df["payment_status"] = df["payment_status"].str.strip().str.lower()
success = df[df["payment_status"] == "success"]
print(success)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  txn_velocity_1hr device_change_flag location_change_flag failed_txn_count_24hr balance_category  \
5           1820.0      3089.0  TXN-1820-3089-13-TXN983778        13.0            60%  2026.0  ...               1.0                 No                   No                   0.0           Normal   
7            546.0      3326.0  TXN-0546-3326-02-TXN991505         2.0            97%  2022.0  ...               0.0                 No                  Yes                   0.0           Normal   
8             86.0      2849.0  TXN-0086-2849-07-TXN990616         7.0            32%  2026.0  ...               1.0                 No                   No                   0.0           Normal   
11             NaN      5062.0  TXN-4857-5062-17-TXN674500        17.0            34%  2025.0  ...               2.0                 No                   No                   0.0            Error   
12   

## Q69 - Convert the amount column to numeric format before comparing with numeric thresholds.

In [71]:
df["account_balance"] = pd.to_numeric(
df["account_balance"],
errors="coerce"
)
highamount = df[df["account_balance"] > 50000]
print(highamount)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  txn_velocity_1hr device_change_flag location_change_flag failed_txn_count_24hr balance_category  \
7359          1.0         NaN  TXN-0001-0000-01-TXN809570         1.0            37%  2021.0  ...               1.0                 No                   No                   0.0           Normal   

     value_group  
7359        High  

[1 rows x 43 columns]


## Q70 - Filter transactions using aml_flag and suspicious_txn_flag with correct logical conditions.

In [72]:
flag = df[
(df["aml_flag"] == 1) &
(df["suspicious_txn_flag"] == 1)
]
print(flag)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr, balance_category, value_group]
Index: []

[0 rows x 43 columns]


## Q71 - Insert a new record into the dataset explicitly specifying all column names.

In [73]:
new_record = {
    "customer_id": 1001,
    "transaction_id": "TXN9999",
    "payment_status": "Success",
    "payment_method": "UPI",
    "account_balance": 25000,
    "Payment_status": "Success",
    "Payment_Method": "Credit_card",
    "Payment_gateway": "PayPal",
    "Credit_Amount": 10000
}
df = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)
print(df.tail())

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  failed_txn_count_24hr balance_category value_group Payment_status Payment_Method Payment_gateway
12998       4164.0      4624.0  TXN-4164-4624-07-TXN823790         7.0            59%  2022.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
12999         18.0      3223.0  TXN-0018-3223-25-TXN692022        25.0            62%  2022.0  ...                    2.0           Normal         Low            NaN            NaN             NaN
13000       3821.0      3210.0  TXN-3821-3210-19-TXN510492        19.0           100%  2024.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
13001       4761.0      3593.0  TXN-4761-3593-17-TXN226172        17.0            90%  2021.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
13002       100

## Q72 - Sort transactions by txn_date in ascending and descending order.

In [74]:
ascdate = df.sort_values(by="txn_date")
print(ascdate)
descdate = df.sort_values(by="txn_date", ascending=False)
print(descdate)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  failed_txn_count_24hr balance_category value_group Payment_status Payment_Method Payment_gateway
9933        1299.0       488.0  TXN-1299-0488-21-TXN492186        21.0            59%  2021.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
2748           NaN      6483.0  TXN-2933-6483-22-TXN802566        22.0            38%  2021.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
5161        4858.0       314.0  TXN-4858-0314-15-TXN252940        15.0            49%  2021.0  ...                    0.0           Normal         NaN            NaN            NaN             NaN
5237        4778.0       859.0  TXN-4778-0859-01-TXN257647         1.0            74%  2021.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
3359        390

## Q73 - Calculate total transactions per customer without unnecessary sorting within groups.

In [75]:
custtxn = df.groupby("customer_id").size()
print(custtxn)

customer_id
1.0       25
4.0        2
5.0        3
6.0        2
7.0        6
          ..
4995.0     1
4996.0     1
4998.0     2
4999.0     6
5000.0     6
Length: 4318, dtype: int64


## Q74 - Select specific columns without including trailing separators.

In [76]:
select = df[["customer_id", "transaction_id", "account_balance"]]
print(select)

       customer_id              transaction_id  account_balance
0           4026.0  TXN-4026-5410-01-TXN278384          8216.04
1           2249.0  TXN-2249-3263-10-TXN997066         -1354.76
2           3542.0  TXN-3542-0133-03-TXN966402           872.32
3            408.0  TXN-0408-6658-08-TXN293594           609.70
4           2990.0  TXN-2990-4126-15-TXN890459          1799.68
...            ...                         ...              ...
12998       4164.0  TXN-4164-4624-07-TXN823790          1254.34
12999         18.0  TXN-0018-3223-25-TXN692022          1640.55
13000       3821.0  TXN-3821-3210-19-TXN510492         10000.00
13001       4761.0  TXN-4761-3593-17-TXN226172          2004.92
13002       1001.0                     TXN9999         25000.00

[13003 rows x 3 columns]


## Q75 - Join transaction records with a customer-level summary keeping all transactions even without matches.

In [77]:
summary = df.groupby("customer_id")["account_balance"].sum().reset_index()
summary.rename(
    columns={"account_balance": "total_balance"},
    inplace=True
)
result = pd.merge(
    df,
    summary,
    on="customer_id",
    how="left"
)
print(result)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway total_balance
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN       8216.04
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN       4684.29
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN       9937.94
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN       2125.26
4           2990.0      4126.0  TXN-2990-4126-15-TXN890

## Q76 - Apply filters after a left join without accidentally converting it into an inner join.

In [78]:
summary = df.groupby("customer_id")["account_balance"].sum().reset_index()
joined = pd.merge(
    df,
    summary,
    on="customer_id",
    how="left"
)
filtered = joined[
    joined["payment_status"] == "Success"
]
print(filtered)

       customer_id  account_id transaction_id  product_id product_margin  year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway account_balance_y
13002       1001.0         NaN        TXN9999         NaN            NaN   NaN  ...               NaN         NaN        Success    Credit_card          PayPal          27073.66

[1 rows x 47 columns]


## Q77 - Join a dataset with itself using unique identifiers like transaction_id.

In [79]:
selfjoin = pd.merge(
    df,
    df,
    on="transaction_id",
    suffixes=("_left", "_right")
)
print(selfjoin)

       customer_id_left  account_id_left              transaction_id  product_id_left product_margin_left  year_left  ...  failed_txn_count_24hr_right balance_category_right value_group_right  \
0                4026.0           5410.0  TXN-4026-5410-01-TXN278384              1.0                 91%     2024.0  ...                          0.0                 Normal               Low   
1                2249.0           3263.0  TXN-2249-3263-10-TXN997066             10.0                 61%     2022.0  ...                          0.0                 Refund               Low   
2                3542.0            133.0  TXN-3542-0133-03-TXN966402              3.0                 35%     2024.0  ...                          0.0                 Normal               Low   
3                 408.0           6658.0  TXN-0408-6658-08-TXN293594              8.0                 38%     2025.0  ...                          0.0                 Normal               Low   
4                2990.0  

## Q78 - Choose the correct join type based on whether missing records should be retained.

In [80]:
summary = df.groupby("customer_id")["reward_points_earned"].sum().reset_index()
result = pd.merge(
    df,
    summary,
    on="customer_id",
    how="left"
)
print(result)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway reward_points_earned_y
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN                 1970.0
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN                  187.0
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN                 5385.0
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN                 1287.0
4         

## Q79 - Apply filtering conditions after joining rather than inside the join logic.

In [81]:
summary = df.groupby("customer_id")["account_balance"].sum().reset_index()
joined = pd.merge(
    df,
    summary,
    on="customer_id",
    how="inner"
)
result = joined[joined["payment_status"] == "Success"]
print(result)

       customer_id  account_id transaction_id  product_id product_margin  year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway account_balance_y
12775       1001.0         NaN        TXN9999         NaN            NaN   NaN  ...               NaN         NaN        Success    Credit_card          PayPal          27073.66

[1 rows x 47 columns]


## Q80 - Build a join using consistent and explicit join methods.

In [82]:
customer_summary = (
    df.groupby("customer_id")["account_balance"]
    .sum()
    .reset_index()
)
joined = pd.merge(
    left=df,
    right=customer_summary,
    on="customer_id",
    how="left"
)
print(joined)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway account_balance_y
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN           8216.04
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN           4684.29
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN           9937.94
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN           2125.26
4           2990.0      4126.0  TXN

## Q81 - Join records using structured numeric IDs rather than text fields.

In [83]:
customer_summary = df.groupby("customer_id")["Credit_Amount"].sum().reset_index()
customer_summary["customer_id"] = pd.to_numeric(
    customer_summary["customer_id"],
    errors="coerce"
)
df["customer_id"] = pd.to_numeric(
    df["customer_id"],
    errors="coerce"
)
result = pd.merge(
    df,
    customer_summary,
    on="customer_id",
    how="left"
)
print(result)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway Credit_Amount_y
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN         2498.96
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN        10068.78
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN        -8610.63
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN        16880.04
4           2990.0      4126.0  TXN-2990-4126

## Q82 - Perform a self-join on customer_id to compare transactions across different dates.

In [84]:
df1 = df[["customer_id", "transaction_id", "txn_date"]].copy()
df2 = df[["customer_id", "transaction_id", "txn_date"]].copy()
res = pd.merge(
    df1,
    df2,
    on="customer_id",
    suffixes=("_1", "_2")
)
res = res[res["txn_date_1"] != res["txn_date_2"]]
print(res)

       customer_id            transaction_id_1 txn_date_1            transaction_id_2 txn_date_2
1           4026.0  TXN-4026-5410-01-TXN278384 2024-10-01  TXN-4026-3511-23-TXN960137 2023-07-20
3           2249.0  TXN-2249-3263-10-TXN997066 2022-07-15  TXN-2249-0469-07-TXN335333 2022-03-16
5           3542.0  TXN-3542-0133-03-TXN966402 2024-02-18  TXN-3542-4992-01-TXN587868 2023-01-27
6           3542.0  TXN-3542-0133-03-TXN966402 2024-02-18  TXN-3542-2371-02-TXN152073 2022-10-14
7           3542.0  TXN-3542-0133-03-TXN966402 2024-02-18  TXN-3542-5015-17-TXN839107 2025-02-22
...            ...                         ...        ...                         ...        ...
98222       3821.0  TXN-3821-3210-19-TXN510492 2024-03-16  TXN-3821-4213-07-TXN740649 2023-10-13
98224       4761.0  TXN-4761-3593-17-TXN226172 2021-04-28  TXN-4761-6468-05-TXN760759        NaT
98225       4761.0  TXN-4761-3593-17-TXN226172 2021-04-28  TXN-4761-5035-16-TXN549404 2026-06-19
98227       1001.0            

## Q83 - Use left join or concatenation instead of a full outer join.

In [85]:
cus_sum = df.groupby("customer_id")["Credit_Amount"].sum().reset_index()
res = pd.merge(
    df,
    cus_sum,
    on="customer_id",
    how="left"
)
print(res)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway Credit_Amount_y
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN         2498.96
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN        10068.78
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN        -8610.63
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN        16880.04
4           2990.0      4126.0  TXN-2990-4126

## Q84 - Join tables using both customer_id and account_id for accurate mapping.

In [86]:
summary = df.groupby(
    ["customer_id", "account_id"]
)["Credit_Amount"].sum().reset_index()
result = pd.merge(
    df,
    summary,
    on=["customer_id", "account_id"],
    how="left"
)
print(result)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway Credit_Amount_y
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN         2406.56
1           2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN           68.78
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN          554.87
3            408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN         1258.98
4           2990.0      4126.0  TXN-2990-4126

## Q85 - Apply filters as separate steps after join operations.

In [87]:
summary = df.groupby("customer_id")["Credit_Amount"].sum().reset_index()
joined = pd.merge(
    df,
    summary,
    on="customer_id",
    how="left"
)
filter = joined[
    joined["payment_status"] == "Success"
]
print(filter)

       customer_id  account_id transaction_id  product_id product_margin  year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway Credit_Amount_y
13002       1001.0         NaN        TXN9999         NaN            NaN   NaN  ...               NaN         NaN        Success    Credit_card          PayPal         12626.4

[1 rows x 47 columns]


## Q86 - Filter transactions within a specific date range after merging datasets.

In [88]:
summary = df.groupby("customer_id")["Credit_Amount"].sum().reset_index()
merged = pd.merge(
    df,
    summary,
    on="customer_id",
    how="left"
)
merged["txn_date"] = pd.to_datetime(
    merged["txn_date"],
    errors="coerce"
)
date = merged[
    (merged["txn_date"] >= "2024-01-01") &
    (merged["txn_date"] <= "2024-12-31")
]
print(date)

       customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway Credit_Amount_y
0           4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN         2498.96
2           3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN        -8610.63
4           2990.0      4126.0  TXN-2990-4126-15-TXN890459        15.0             9%  2024.0  ...            Normal         Low            NaN            NaN             NaN        29811.97
17          3124.0      7575.0  TXN-3124-7575-08-TXN115920         8.0            74%  2024.0  ...            Normal         Low            NaN            NaN             NaN         9536.56
28           927.0      3113.0  TXN-0927-3113

## Q87 - Ensure join columns have matching data types before performing joins.

In [89]:
df["customer_id"] = df["customer_id"].astype(str)
summary = df.groupby("customer_id")["Credit_Amount"].sum().reset_index()
summary["customer_id"] = summary["customer_id"].astype(str)
result = pd.merge(
    df,
    summary,
    on="customer_id"
)
print(result)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway Credit_Amount_y
0          4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN         2498.96
1          2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN        10068.78
2          3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN        -8610.63
3           408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN        16880.04
4          2990.0      4126.0  TXN-2990-4126-15-TX

## Q88 - Handle NULL values in join keys before performing merges.

In [90]:
df_clean = df[df["customer_id"].notnull()]
summary = (
    df_clean.groupby("customer_id")["account_balance"]
    .sum()
    .reset_index()
)
result = pd.merge(
    df_clean,
    summary,
    on="customer_id",
    how="left"
)
print(result)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway account_balance_y
0          4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN           8216.04
1          2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN           4684.29
2          3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN           9937.94
3           408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN           2125.26
4          2990.0      4126.0  TXN-2990-

## Q89 - Avoid joining the same dataset multiple times unnecessarily.

In [91]:
sum = df.groupby("customer_id").agg({
    "account_balance": "sum",
    "reward_points_earned": "sum"
}).reset_index()
join = pd.merge(
    df,
    sum,
    on="customer_id",
    how="left"
)
print(join)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  value_group Payment_status Payment_Method Payment_gateway account_balance_y reward_points_earned_y
0          4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...          Low            NaN            NaN             NaN           8216.04                 1970.0
1          2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...          Low            NaN            NaN             NaN           4684.29                  187.0
2          3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...          Low            NaN            NaN             NaN           9937.94                 5385.0
3           408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...          Low            NaN            NaN             NaN           2125.26                 1287.0
4         

## Q90 - Apply filters at the correct stage instead of repeating the same filter across multiple operations.

In [92]:
success = df[
    df["payment_status"].str.strip().str.lower() == "success"
]
customer_sum = success.groupby("customer_id").agg(
    total_amount=("Credit_Amount", "sum"),
    transaction_count=("Credit_Amount", "count")
).reset_index()
result = pd.merge(
    success,
    customer_sum,
    on="customer_id",
    how="left"
)
print(result)

     customer_id  account_id              transaction_id  product_id product_margin    year  ...  value_group Payment_status Payment_Method Payment_gateway total_amount transaction_count
0         1820.0      3089.0  TXN-1820-3089-13-TXN983778        13.0            60%  2026.0  ...          Low            NaN            NaN             NaN     22107.03               3.0
1          546.0      3326.0  TXN-0546-3326-02-TXN991505         2.0            97%  2022.0  ...          Low            NaN            NaN             NaN     10263.64               3.0
2           86.0      2849.0  TXN-0086-2849-07-TXN990616         7.0            32%  2026.0  ...          Low            NaN            NaN             NaN      3256.91               1.0
3            NaN      5062.0  TXN-4857-5062-17-TXN674500        17.0            34%  2025.0  ...          Low            NaN            NaN             NaN          NaN               NaN
4         3855.0      6648.0  TXN-3855-6648-15-TXN722333        1

## Q91 - Use appropriate merge strategies when combining customer-level and transaction-level data.

In [93]:
customer_summary = df.groupby("customer_id")["Credit_Amount"].sum().reset_index()
result = pd.merge(
    df,
    customer_summary,
    on="customer_id",
    how="left"
)
print(result)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway Credit_Amount_y
0          4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...            Normal         Low            NaN            NaN             NaN         2498.96
1          2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...            Refund         Low            NaN            NaN             NaN        10068.78
2          3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...            Normal         Low            NaN            NaN             NaN        -8610.63
3           408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...            Normal         Low            NaN            NaN             NaN        16880.04
4          2990.0      4126.0  TXN-2990-4126-15-TX

## Q92 - Assign meaningful names to intermediate datasets.

In [94]:
successful_transactions = df[
    df["payment_status"].str.strip().str.lower() == "success"
]
customer_transaction_summary = successful_transactions.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
print(customer_transaction_summary)

     customer_id  Credit_Amount
0            1.0       30490.51
1           10.0         624.92
2          100.0        8922.86
3         1000.0       26164.24
4         1001.0       12626.40
...          ...            ...
2216       989.0        4384.21
2217       990.0       10000.00
2218       992.0       14620.51
2219       993.0        -102.51
2220       994.0        1409.35

[2221 rows x 2 columns]


## Q93 - Select columns explicitly to avoid accidentally using same-named columns from different sources.

In [95]:
transaction = df[
    ["customer_id", "account_id", "transaction_id",
     "txn_date", "txn_type", "payment_status"]
]
customer = df[
    ["customer_id", "region", "customer_segment"]
].drop_duplicates("customer_id")
result = pd.merge(
    transaction,
    customer,
    on="customer_id",
    how="left"
)
print(result)

      customer_id  account_id              transaction_id   txn_date    txn_type payment_status      region customer_segment
0          4026.0      5410.0  TXN-4026-5410-01-TXN278384 2024-10-01  Withdrawal        pending        east             Gold
1          2249.0      3263.0  TXN-2249-3263-10-TXN997066 2022-07-15      Refund        pending  south-west         Platinum
2          3542.0       133.0  TXN-3542-0133-03-TXN966402 2024-02-18      Refund       refunded          w2        Corporate
3           408.0      6658.0  TXN-0408-6658-08-TXN293594 2025-02-04     Deposit       refunded  South-East         Platinum
4          2990.0      4126.0  TXN-2990-4126-15-TXN890459 2024-04-17    Purchase       refunded      enorth             Gold
...           ...         ...                         ...        ...         ...            ...         ...              ...
12998      4164.0      4624.0  TXN-4164-4624-07-TXN823790 2022-09-15  Withdrawal        success  North-East             Gold


## Q94 - Remove unnecessary joins that do not change the result row count.

In [96]:
customer = df[
    ["customer_id", "region"]
].drop_duplicates("customer_id")
result = pd.merge(
    df,
    customer,
    on="customer_id",
    how="left"
)
print("Original rows:", len(df))
print("Rows after join:", len(result))

Original rows: 13003
Rows after join: 13003


## Q95 - Select only required columns instead of keeping all columns.

In [97]:
result = df[
    ["customer_id", "account_id", "transaction_id",
     "txn_date", "region", "txn_type",
     "payment_method", "payment_status", "Credit_Amount"]
]
print(result)

      customer_id  account_id              transaction_id   txn_date      region    txn_type payment_method payment_status  Credit_Amount
0          4026.0      5410.0  TXN-4026-5410-01-TXN278384 2024-10-01        east  Withdrawal            UPI        pending        2406.56
1          2249.0      3263.0  TXN-2249-3263-10-TXN997066 2022-07-15  south-west      Refund    Net Banking        pending          68.78
2          3542.0       133.0  TXN-3542-0133-03-TXN966402 2024-02-18          w2      Refund            UPI       refunded         554.87
3           408.0      6658.0  TXN-0408-6658-08-TXN293594 2025-02-04  South-East     Deposit         Wallet       refunded        1258.98
4          2990.0      4126.0  TXN-2990-4126-15-TXN890459 2024-04-17      enorth    Purchase    Credit Card       refunded       10000.00
...           ...         ...                         ...        ...         ...         ...            ...            ...            ...
12998      4164.0      4624.0  TXN

## Q96 - Filter payment_method using pattern matching without leading wildcards.

In [98]:
result = df[
    df["payment_method"].str.strip().str.startswith(
        "Credit", na=False
    )
]
print(result)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  failed_txn_count_24hr balance_category value_group Payment_status Payment_Method Payment_gateway
4          2990.0      4126.0  TXN-2990-4126-15-TXN890459        15.0             9%  2024.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
6          2762.0      4749.0  TXN-2762-4749-07-TXN585525         7.0            45%  2022.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
8            86.0      2849.0  TXN-0086-2849-07-TXN990616         7.0            32%  2026.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
10          898.0      5120.0  TXN-0898-5120-21-TXN325000        21.0            NaN  2026.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
11            NaN   

## Q97 - Filter transactions using direct date comparisons instead of extracting and comparing date components.

In [99]:
df["txn_date"] = pd.to_datetime(
    df["txn_date"],
    errors="coerce"
)
result = df[
    (df["txn_date"] >= "2026-01-01") &
    (df["txn_date"] <= "2026-06-30")
]
print(result)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  failed_txn_count_24hr balance_category value_group Payment_status Payment_Method Payment_gateway
5          1820.0      3089.0  TXN-1820-3089-13-TXN983778        13.0            60%  2026.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
8            86.0      2849.0  TXN-0086-2849-07-TXN990616         7.0            32%  2026.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
15         2524.0      3600.0  TXN-2524-3600-11-TXN540644        11.0            54%  2026.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
31         1897.0       966.0  TXN-1897-0966-21-TXN119920        21.0            10%  2026.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
34         3833.0   

## Q98 - Combine two datasets row-wise without removing duplicates when not needed.

In [100]:
first_data = df.iloc[:100]
second_data = df.iloc[100:200]
result = pd.concat(
    [first_data, second_data],
    ignore_index=True
)
print(result)

    customer_id  account_id              transaction_id  product_id product_margin    year  ...  failed_txn_count_24hr balance_category value_group Payment_status Payment_Method Payment_gateway
0        4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
1        2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...                    0.0           Refund         Low            NaN            NaN             NaN
2        3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
3         408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...                    0.0           Normal         Low            NaN            NaN             NaN
4        2990.0      4126.0  T

## Q99 - Add aggregated values (like customer totals) as a new column to the original transaction data.

In [101]:
df["customer_total_amount"] = df.groupby(
    "customer_id"
)["Credit_Amount"].transform("sum")
print(
    df[
        ["customer_id", "transaction_id",
         "Credit_Amount", "customer_total_amount"]
    ]
)

      customer_id              transaction_id  Credit_Amount  customer_total_amount
0          4026.0  TXN-4026-5410-01-TXN278384        2406.56                2498.96
1          2249.0  TXN-2249-3263-10-TXN997066          68.78               10068.78
2          3542.0  TXN-3542-0133-03-TXN966402         554.87               -8610.63
3           408.0  TXN-0408-6658-08-TXN293594        1258.98               16880.04
4          2990.0  TXN-2990-4126-15-TXN890459       10000.00               29811.97
...           ...                         ...            ...                    ...
12998      4164.0  TXN-4164-4624-07-TXN823790         -59.64               11659.60
12999        18.0  TXN-0018-3223-25-TXN692022        1940.12                1252.93
13000      3821.0  TXN-3821-3210-19-TXN510492       10000.00               13567.39
13001      4761.0  TXN-4761-3593-17-TXN226172         298.86                 172.69
13002      1001.0                     TXN9999       10000.00               1

## Q100 - Simplify mathematical conditions (e.g., instead of amount / 2 > threshold, rewrite as amount > threshold * 2).

In [102]:
df["Credit_Amount"] = pd.to_numeric(
    df["Credit_Amount"],
    errors="coerce"
)
result = df[
    df["Credit_Amount"] > 25000 * 2
]
print(result)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr, balance_category, value_group, Payment_status, Payment_Method, Payment_gateway, customer_total_amount]
Index: []

[0 rows x 47 columns]


## Q101 - Analyze complex filtering logic step by step instead of trial and error.

In [103]:
success = df[df["payment_status"] == "Success"]
highvalue = success[
    success["Credit_Amount"] > 50000
]
result = highvalue[
    highvalue["fraud_risk_level"] == "High"
]
print(result)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr, balance_category, value_group, Payment_status, Payment_Method, Payment_gateway, customer_total_amount]
Index: []

[0 rows x 47 columns]


## Q102 - Break complex nested operations into simpler intermediate steps.

In [104]:
activecust = df[
    df["customer_activity_status"] == "Active"
]
successfultrans = activecust[
    activecust["payment_status"] == "Success"
]
result = successfultrans[
    successfultrans["Credit_Amount"] > 10000
]
print(result)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr, balance_category, value_group, Payment_status, Payment_Method, Payment_gateway, customer_total_amount]
Index: []

[0 rows x 47 columns]


## Q103 - Apply filters on high-cardinality columns first before other conditions.

In [105]:
customerdata = df[
    df["customer_id"] == 4026
]
result = customerdata[
    customerdata["payment_status"] == "Success"
]
print(result)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr, balance_category, value_group, Payment_status, Payment_Method, Payment_gateway, customer_total_amount]
Index: []

[0 rows x 47 columns]


## Q104 - Replace long chains of OR conditions with a single membership check.

In [106]:
result = df[
    df["payment_status"].isin(
        ["Success", "Pending", "Refunded"]
    )
]
print(result)

      customer_id  account_id transaction_id  product_id product_margin  year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway customer_total_amount
13002      1001.0         NaN        TXN9999         NaN            NaN   NaN  ...               NaN         NaN        Success    Credit_card          PayPal               12626.4

[1 rows x 47 columns]


## Q105 - Avoid creating unnecessary copies of data during analysis.

In [107]:
res = df[
    ["customer_id", "transaction_id",
     "payment_status", "Credit_Amount"]
]
res = res[
    res["payment_status"] == "Success"
]
print(res)

      customer_id transaction_id payment_status  Credit_Amount
13002      1001.0        TXN9999        Success        10000.0


## Q106 - Sort data using a single simple column where possible.

In [108]:
result = df.sort_values(
    "Credit_Amount",
    ascending=False
)
print(result)

      customer_id  account_id               transaction_id  product_id product_margin    year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway customer_total_amount
1255       4302.0      4334.0   TXN-4302-4334-14-TXN727959        14.0            36%  2022.0  ...            Normal         Low            NaN            NaN             NaN              26076.42
65         1000.0      5848.0   TXN-1000-5848-23-TXN796992        23.0            13%  2025.0  ...            Normal         Low            NaN            NaN             NaN              50201.61
1951       2246.0      6500.0   TXN-2246-6500-10-TXN335291        10.0            81%  2023.0  ...            Normal         Low            NaN            NaN             NaN              47705.88
7183       1686.0      6425.0   TXN-1686-6425-07-TXN724370         7.0             8%  2023.0  ...            Normal         Low            NaN            NaN             NaN              16354.69
6268       3408

## Q107 - Replace multiple OR conditions with a single IN membership test.

In [109]:
result = df[
    df["txn_type"].isin(
        ["Deposit", "Purchase", "Withdrawal"]
    )
]
print("matching rows:", len(result))
print(result[["customer_id", "txn_type"]].head(10).to_string(index=False))


matching rows: 9546
customer_id   txn_type
     4026.0 Withdrawal
      408.0    Deposit
     2990.0   Purchase
     1820.0 Withdrawal
     2762.0    Deposit
      546.0    Deposit
       86.0   Purchase
     4347.0   Purchase
      898.0   Purchase
        NaN    Deposit


## Q108 - Compute aggregations once and reuse the result.

In [110]:
customer_summary = df.groupby("customer_id").agg(
    total_amount=("Credit_Amount", "sum"),
    transaction_count=("transaction_id", "count")
)
print(customer_summary)
topcust = customer_summary[
    customer_summary["total_amount"] > 100000
]
print(topcust)

             total_amount  transaction_count
customer_id                                 
1.0              94428.30                 25
10.0             15454.30                  3
100.0             5891.89                  2
1000.0           50201.61                  6
1001.0           12626.40                  2
...                   ...                ...
992.0            21199.22                  5
993.0             -102.51                  1
994.0            18288.15                  4
997.0            10000.00                  1
998.0             1806.33                  1

[4318 rows x 2 columns]
Empty DataFrame
Columns: [total_amount, transaction_count]
Index: []


## Q109 - Store derived datasets after a single computation and reuse instead of recomputing.

In [111]:
successful_transactions = df[
    df["payment_status"] == "Success"
]
total_successful = successful_transactions[
    "Credit_Amount"
].sum()
average_successful = successful_transactions[
    "Credit_Amount"
].mean()
print("Total:", total_successful)
print("Average:", average_successful)

Total: 10000.0
Average: 10000.0


## Q110 - Avoid applying text transformations on filter columns unnecessarily.

In [112]:
res = df[
    df["payment_status"] == "Success"
]
print(res)

      customer_id  account_id transaction_id  product_id product_margin  year  ...  balance_category value_group Payment_status Payment_Method Payment_gateway customer_total_amount
13002      1001.0         NaN        TXN9999         NaN            NaN   NaN  ...               NaN         NaN        Success    Credit_card          PayPal               12626.4

[1 rows x 47 columns]


## Q111 - Remove temporary intermediate datasets when no longer needed

In [113]:
successful_transactions = df[
    df["payment_status"] == "Success"
]
result = successful_transactions.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
del successful_transactions
print(result)

  customer_id  Credit_Amount
0      1001.0        10000.0


## Q112 - Perform all filtering within data operations instead of using external loops

In [114]:
result = df[
    (df["payment_status"] == "Success") &
    (df["Credit_Amount"] > 10000) &
    (df["txn_type"].isin(["Deposit", "Purchase"]))
]
print(result)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr, balance_category, value_group, Payment_status, Payment_Method, Payment_gateway, customer_total_amount]
Index: []

[0 rows x 47 columns]


## Q113 - Use merge operations instead of iterating row-by-row to combine data

In [115]:
custsummary = df.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
custsummary = custsummary.rename(
    columns={"Credit_Amount": "total_amount"}
)
result = pd.merge(
    df,
    custsummary,
    on="customer_id",
    how="left"
)
print(result)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  value_group Payment_status Payment_Method Payment_gateway customer_total_amount total_amount
0          4026.0      5410.0  TXN-4026-5410-01-TXN278384         1.0            91%  2024.0  ...          Low            NaN            NaN             NaN               2498.96      2498.96
1          2249.0      3263.0  TXN-2249-3263-10-TXN997066        10.0            61%  2022.0  ...          Low            NaN            NaN             NaN              10068.78     10068.78
2          3542.0       133.0  TXN-3542-0133-03-TXN966402         3.0            35%  2024.0  ...          Low            NaN            NaN             NaN              -8610.63     -8610.63
3           408.0      6658.0  TXN-0408-6658-08-TXN293594         8.0            38%  2025.0  ...          Low            NaN            NaN             NaN              16880.04     16880.04
4          2990.0      4126.0  TXN-2990-

## Q114 - Ensure consistent data types across columns before performing operations

In [116]:
df["customer_id"] = df["customer_id"].astype(str)
df["Credit_Amount"] = pd.to_numeric(
    df["Credit_Amount"],
    errors="coerce"
)
df["txn_date"] = pd.to_datetime(
    df["txn_date"],
    errors="coerce"
)
print(df.dtypes)

customer_id                   str
account_id                float64
transaction_id                str
product_id                float64
product_margin                str
                           ...   
value_group              category
Payment_status                str
Payment_Method                str
Payment_gateway               str
customer_total_amount     float64
Length: 47, dtype: object


## Q115 - Calculate total transaction amount per customer correctly without averaging already-averaged values

In [117]:
result = df.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
result = result.rename(
    columns={"Credit_Amount": "total_transaction_amount"}
)
print(result)

     customer_id  total_transaction_amount
0            1.0                  94428.30
1           10.0                  15454.30
2          100.0                   5891.89
3         1000.0                  50201.61
4         1001.0                  12626.40
...          ...                       ...
4313       992.0                  21199.22
4314       993.0                   -102.51
4315       994.0                  18288.15
4316       997.0                  10000.00
4317       998.0                   1806.33

[4318 rows x 2 columns]


## Q116 - Apply row-level filters before aggregation, not after

In [118]:
successful_trans = df[
    df["payment_status"] == "Success"
]
result = successful_trans.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
print(result)

  customer_id  Credit_Amount
0      1001.0        10000.0


## Q117 - Count unique customers in the dataset

In [119]:
uniquecust = df["customer_id"].nunique()
print("Number of unique customers:", uniquecust)

Number of unique customers: 4318


## Q118 - Create running totals or moving averages for transactions per customer

In [120]:
df["txn_date"] = pd.to_datetime(
    df["txn_date"],
    errors="coerce"
)
df = df.sort_values(
    ["customer_id", "txn_date"]
)
df["running_total"] = df.groupby(
    "customer_id"
)["Credit_Amount"].cumsum()
df["moving_average"] = df.groupby(
    "customer_id"
)["Credit_Amount"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
print(
    df[
        ["customer_id", "txn_date",
         "Credit_Amount", "running_total",
         "moving_average"]
    ]
)

      customer_id   txn_date  Credit_Amount  running_total  moving_average
671           1.0 2021-05-12       10000.00       10000.00    10000.000000
8090          1.0 2021-10-22       10000.00       20000.00    10000.000000
7359          1.0 2021-12-15        3950.24       23950.24     7983.413333
11748         1.0 2021-12-15        3950.24       27900.48     5966.826667
2240          1.0 2022-03-01        2231.78       30132.26     3377.420000
...           ...        ...            ...            ...             ...
7613          NaN        NaT         687.57            NaN             NaN
9019          NaN        NaT        1057.07            NaN             NaN
9044          NaN        NaT        1814.52            NaN             NaN
9103          NaN        NaT       10000.00            NaN             NaN
11587         NaN        NaT        6279.66            NaN             NaN

[13003 rows x 5 columns]


## Q119 - Rank customers by a specific metric

In [121]:
custtotal = df.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
custtotal["rank"] = custtotal[
    "Credit_Amount"
].rank(
    method="dense",
    ascending=False
)
customer_total = custtotal.sort_values("rank")
print(customer_total)

     customer_id  Credit_Amount    rank
0            1.0       94428.30     1.0
3203      4338.0       56908.39     2.0
510       1546.0       56506.70     3.0
1890      2989.0       54169.22     4.0
3392      4532.0       53623.05     5.0
...          ...            ...     ...
2178      3284.0      -19105.85  4041.0
3679      4830.0      -19302.30  4042.0
615       1652.0      -20440.08  4043.0
1234      2307.0      -25957.56  4044.0
3569      4718.0      -26326.00  4045.0

[4318 rows x 3 columns]


## Q120 - Generate subtotal reports with multiple aggregation functions

In [122]:
result = df.groupby(
    "customer_id"
).agg(
    total_amount=("Credit_Amount", "sum"),
    average_amount=("Credit_Amount", "mean"),
    highest_amount=("Credit_Amount", "max"),
    lowest_amount=("Credit_Amount", "min"),
    transaction_count=("transaction_id", "count")
).reset_index()
print(result)

     customer_id  total_amount  average_amount  highest_amount  lowest_amount  transaction_count
0            1.0      94428.30     3777.132000        10000.00      -10510.67                 25
1           10.0      15454.30     5151.433333        10000.00         624.92                  3
2          100.0       5891.89     2945.945000         8922.86       -3030.97                  2
3         1000.0      50201.61     8366.935000        23927.30        1118.47                  6
4         1001.0      12626.40     6313.200000        10000.00        2626.40                  2
...          ...           ...             ...             ...            ...                ...
4313       992.0      21199.22     4239.844000         7363.25         530.46                  5
4314       993.0       -102.51     -102.510000         -102.51        -102.51                  1
4315       994.0      18288.15     3657.630000        10000.00         755.30                  4
4316       997.0      10000.00

## Q121 - Keep raw transaction data and aggregated summaries in separate variables

In [123]:
raw_transactions = df
customer_summary = df.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
print(raw_transactions)
print(customer_summary)

      customer_id  account_id               transaction_id  product_id product_margin    year  ...  Payment_status Payment_Method Payment_gateway customer_total_amount running_total moving_average
671           1.0         NaN   TXN-0001-0000-01-TXN206851         1.0            40%  2021.0  ...             NaN            NaN             NaN               94428.3      10000.00   10000.000000
8090          1.0         NaN   TXN-0001-0000-01-TXN627276         1.0            99%  2021.0  ...             NaN            NaN             NaN               94428.3      20000.00   10000.000000
7359          1.0         NaN   TXN-0001-0000-01-TXN809570         1.0            37%  2021.0  ...             NaN            NaN             NaN               94428.3      23950.24    7983.413333
11748         1.0         NaN  TXN-0001-0000-01-TXN1000666         1.0            31%  2021.0  ...             NaN            NaN             NaN               94428.3      27900.48    5966.826667
2240          1

## Q122 - Rank transactions within each customer group

In [124]:
df["transaction_rank"] = df.groupby(
    "customer_id"
)["Credit_Amount"].rank(
    method="dense",
    ascending=False
)
print(
    df[
        ["customer_id", "transaction_id",
         "Credit_Amount", "transaction_rank"]
    ]
)

      customer_id               transaction_id  Credit_Amount  transaction_rank
671           1.0   TXN-0001-0000-01-TXN206851       10000.00               1.0
8090          1.0   TXN-0001-0000-01-TXN627276       10000.00               1.0
7359          1.0   TXN-0001-0000-01-TXN809570        3950.24               4.0
11748         1.0  TXN-0001-0000-01-TXN1000666        3950.24               4.0
2240          1.0   TXN-0001-0000-01-TXN322086        2231.78               7.0
...           ...                          ...            ...               ...
7613          NaN   TXN-2761-6974-13-TXN139444         687.57               NaN
9019          NaN   TXN-1490-5513-20-TXN680820        1057.07               NaN
9044          NaN   TXN-0773-6778-22-TXN734409        1814.52               NaN
9103          NaN   TXN-3157-1287-20-TXN232264       10000.00               NaN
11587         NaN   TXN-3601-5071-18-TXN596972        6279.66               NaN

[13003 rows x 4 columns]


## Q123 - Handle NULL values explicitly in aggregation calculations

In [125]:
df["Credit_Amount"] = pd.to_numeric(
    df["Credit_Amount"],
    errors="coerce"
)
customer_total = df.groupby(
    "customer_id"
)["Credit_Amount"].sum(min_count=1).reset_index()
print(customer_total)

     customer_id  Credit_Amount
0            1.0       94428.30
1           10.0       15454.30
2          100.0        5891.89
3         1000.0       50201.61
4         1001.0       12626.40
...          ...            ...
4313       992.0       21199.22
4314       993.0        -102.51
4315       994.0       18288.15
4316       997.0       10000.00
4317       998.0        1806.33

[4318 rows x 2 columns]


## Q124 - Filter the results of aggregated or ranked data

In [126]:
customer_summary = df.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
customer_summary = customer_summary[
    customer_summary["Credit_Amount"] > 50000
]
print(customer_summary)

     customer_id  Credit_Amount
0            1.0       94428.30
3         1000.0       50201.61
510       1546.0       56506.70
1089      2163.0       52167.12
1890      2989.0       54169.22
2155      3262.0       50115.07
3203      4338.0       56908.39
3392      4532.0       53623.05


## Q125 - Apply sorting only at the final output stage

In [127]:
customer_summary = df.groupby(
    "customer_id"
).agg(
    total_amount=("Credit_Amount", "sum"),
    transaction_count=("transaction_id", "count")
).reset_index()
result = customer_summary.sort_values(
    "total_amount",
    ascending=False
)
print(result)

     customer_id  total_amount  transaction_count
0            1.0      94428.30                 25
3203      4338.0      56908.39                  6
510       1546.0      56506.70                  7
1890      2989.0      54169.22                  6
3392      4532.0      53623.05                  7
...          ...           ...                ...
2178      3284.0     -19105.85                  2
3679      4830.0     -19302.30                  3
615       1652.0     -20440.08                  4
1234      2307.0     -25957.56                  2
3569      4718.0     -26326.00                  7

[4318 rows x 3 columns]


## Q126 - Convert integer columns to appropriate types for accurate ratio calculations

In [128]:
success_count = (
    df["payment_status"] == "Success"
).sum()
total_count = len(df)
success_ratio = (
    float(success_count) / float(total_count)
)
print("Success ratio:", success_ratio)

Success ratio: 7.690532953933707e-05


## Q127 - Generate a complete date series and fill in missing dates

In [129]:
df["txn_date"] = pd.to_datetime(
    df["txn_date"],
    errors="coerce"
)
daily_transactions = df.groupby(
    "txn_date"
).size()
all_dates = pd.date_range(
    start=daily_transactions.index.min(),
    end=daily_transactions.index.max(),
    freq="D"
)
daily_transactions = daily_transactions.reindex(
    all_dates,
    fill_value=0
)
print(daily_transactions)

2021-01-01    7
2021-01-02    4
2021-01-03    5
2021-01-04    8
2021-01-05    4
             ..
2026-12-24    6
2026-12-25    9
2026-12-26    4
2026-12-27    6
2026-12-28    7
Freq: D, Length: 2188, dtype: int64


## Q128 - Apply max and min correctly on numeric fields

In [130]:
df["Credit_Amount"] = pd.to_numeric(
    df["Credit_Amount"],
    errors="coerce"
)
maximum_amount = df["Credit_Amount"].max()
minimum_amount = df["Credit_Amount"].min()
print("Maximum amount:", maximum_amount)
print("Minimum amount:", minimum_amount)

Maximum amount: 24078.53
Minimum amount: -31465.75


## Q129 - Reset grouping indexes to convert them back to regular columns

In [131]:
customer_summary = df.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
print(customer_summary)

     customer_id  Credit_Amount
0            1.0       94428.30
1           10.0       15454.30
2          100.0        5891.89
3         1000.0       50201.61
4         1001.0       12626.40
...          ...            ...
4313       992.0       21199.22
4314       993.0        -102.51
4315       994.0       18288.15
4316       997.0       10000.00
4317       998.0        1806.33

[4318 rows x 2 columns]


## Q130 - Load a fresh copy of data, apply changes, and save the final version

In [132]:
freshdata = pd.read_excel(DATA)
freshdata["Credit_Amount"] = pd.to_numeric(
    freshdata["Credit_Amount"],
    errors="coerce"
)
freshdata["payment_method"] = (
    freshdata["payment_method"].str.strip()
)
freshdata.to_excel(
    OUTDIR / "Bank Records 1_cleaned.xlsx",
    index=False
)
print("Cleaned data saved successfully.")

Cleaned data saved successfully.


## Q131 - Perform analysis on a copy of production data without modifying the original

In [133]:
analysis_data = df.copy()
analysis_data["Credit_Amount"] = pd.to_numeric(
    analysis_data["Credit_Amount"],
    errors="coerce"
)
result = analysis_data.groupby(
    "customer_id"
)["Credit_Amount"].sum().reset_index()
print(result)

     customer_id  Credit_Amount
0            1.0       94428.30
1           10.0       15454.30
2          100.0        5891.89
3         1000.0       50201.61
4         1001.0       12626.40
...          ...            ...
4313       992.0       21199.22
4314       993.0        -102.51
4315       994.0       18288.15
4316       997.0       10000.00
4317       998.0        1806.33

[4318 rows x 2 columns]


## Q132 - Validate data after multiple operations and save the final version

In [134]:
final_data = df.copy()
final_data["payment_method"] = (
    final_data["payment_method"].str.strip()
)
final_data["Credit_Amount"] = pd.to_numeric(
    final_data["Credit_Amount"],
    errors="coerce"
)
final_data["txn_date"] = pd.to_datetime(
    final_data["txn_date"],
    errors="coerce"
)
print("Rows:", len(final_data))
print("Duplicates:", final_data.duplicated().sum())
print("NULL values:", final_data.isnull().sum().sum())
final_data.to_excel(
    OUTDIR / "Bank Records 1_final.xlsx",
    index=False
)

Rows: 13003
Duplicates: 0
NULL values: 57332


## Q133 - Filter using NOT IN logic while properly handling NULL values

In [135]:
excluded_status = ["Failed", "Refunded"]
result = df[
    (~df["payment_status"].isin(excluded_status)) &
    (df["payment_status"].notna())
]
print(result)

      customer_id  account_id               transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
671           1.0         NaN   TXN-0001-0000-01-TXN206851         1.0            40%  2021.0  ...             NaN             NaN               94428.3      10000.00   10000.000000              1.0
7359          1.0         NaN   TXN-0001-0000-01-TXN809570         1.0            37%  2021.0  ...             NaN             NaN               94428.3      23950.24    7983.413333              4.0
11748         1.0         NaN  TXN-0001-0000-01-TXN1000666         1.0            31%  2021.0  ...             NaN             NaN               94428.3      27900.48    5966.826667              4.0
2240          1.0         NaN   TXN-0001-0000-01-TXN322086         1.0             3%  2022.0  ...             NaN             NaN               94428.3      30132.26    3377.420000              7.0
3128 

## Q134 - Use consistent datetime handling when filtering across time periods

In [136]:
df["txn_date"] = pd.to_datetime(
    df["txn_date"],
    errors="coerce"
)
start_date = pd.Timestamp("2026-01-01")
end_date = pd.Timestamp("2026-06-30")
result = df[
    (df["txn_date"] >= start_date) &
    (df["txn_date"] <= end_date)
]
print(result)

      customer_id  account_id              transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
9937       1000.0      5373.0  TXN-1000-5373-25-TXN478242        25.0           100%  2026.0  ...             NaN             NaN              50201.61      45156.94   14306.666667              3.0
3191       1001.0      3338.0  TXN-1001-3338-21-TXN854612        21.0            12%  2026.0  ...             NaN             NaN              12626.40       2626.40    2626.400000              2.0
368        1005.0       240.0  TXN-1005-0240-07-TXN768292         7.0             4%  2026.0  ...             NaN             NaN              13421.13      13278.74    4426.246667              3.0
1408       1005.0       830.0  TXN-1005-0830-16-TXN775751        16.0            27%  2026.0  ...             NaN             NaN              13421.13           NaN    5142.405000              NaN
12481     

## Q135 - Validate string lengths before assigning new value

In [137]:
new_payment_method = "Credit Card"
valid_rows = df["payment_method"].str.len().fillna(0) <= len(
    new_payment_method
)
df.loc[valid_rows, "payment_method"] = new_payment_method
print(df["payment_method"])

671      Credit Card
8090     Credit Card
7359     Credit Card
11748    Credit Card
2240     Credit Card
            ...     
7613     Credit Card
9019     Credit Card
9044     Credit Card
9103     Credit Card
11587    Credit Card
Name: payment_method, Length: 13003, dtype: str


## Q136 - Apply updates in a consistent order

In [138]:
updated_data = df.copy()
updated_data["payment_method"] = (
    updated_data["payment_method"].str.strip()
)
updated_data["Credit_Amount"] = pd.to_numeric(
    updated_data["Credit_Amount"],
    errors="coerce"
)
updated_data["txn_date"] = pd.to_datetime(
    updated_data["txn_date"],
    errors="coerce"
)
print(updated_data)

      customer_id  account_id               transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
671           1.0         NaN   TXN-0001-0000-01-TXN206851         1.0            40%  2021.0  ...             NaN             NaN               94428.3      10000.00   10000.000000              1.0
8090          1.0         NaN   TXN-0001-0000-01-TXN627276         1.0            99%  2021.0  ...             NaN             NaN               94428.3      20000.00   10000.000000              1.0
7359          1.0         NaN   TXN-0001-0000-01-TXN809570         1.0            37%  2021.0  ...             NaN             NaN               94428.3      23950.24    7983.413333              4.0
11748         1.0         NaN  TXN-0001-0000-01-TXN1000666         1.0            31%  2021.0  ...             NaN             NaN               94428.3      27900.48    5966.826667              4.0
2240 

## Q137 - Avoid modifying data row by row using iterative loops

In [139]:
result = df[
    (df["payment_status"] == "Success") &
    (df["Credit_Amount"] > 10000)
]
print(result)

Empty DataFrame
Columns: [customer_id, account_id, transaction_id, product_id, product_margin, year, month, region, customer_segment, kyc_status, txn_id_external, txn_type, txn_mode, payment_method, payment_gateway, payment_status, device_type, app_version, offer_id, offer_category, offer_applied_flag, reward_points_earned, offer_effectiveness, account_balance, Fixed_Deposite, Credit_Amount, txn_date, fraud_risk_level, aml_flag, suspicious_txn_flag, clv, avg_transaction_value, txn_frequency_30d, customer_activity_status, most_common_txn_time_of_day, averag_high_txn_weekend_flag, averag_high_txn_day_of_week, txn_velocity_1hr, device_change_flag, location_change_flag, failed_txn_count_24hr, balance_category, value_group, Payment_status, Payment_Method, Payment_gateway, customer_total_amount, running_total, moving_average, transaction_rank]
Index: []

[0 rows x 50 columns]


## Q138 - Validate constraints before applying changes

In [140]:
duplicate_keys = df["transaction_id"].duplicated().sum()
missing_customer_ids = df["customer_id"].isna().sum()
print("Duplicate transaction IDs:", duplicate_keys)
print("Missing customer IDs:", missing_customer_ids)
valid_data = df[
    df["transaction_id"].notna() &
    df["customer_id"].notna()
].drop_duplicates("transaction_id")
print(valid_data)

Duplicate transaction IDs: 236
Missing customer IDs: 227
      customer_id  account_id               transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
671           1.0         NaN   TXN-0001-0000-01-TXN206851         1.0            40%  2021.0  ...             NaN             NaN              94428.30      10000.00   10000.000000              1.0
8090          1.0         NaN   TXN-0001-0000-01-TXN627276         1.0            99%  2021.0  ...             NaN             NaN              94428.30      20000.00   10000.000000              1.0
7359          1.0         NaN   TXN-0001-0000-01-TXN809570         1.0            37%  2021.0  ...             NaN             NaN              94428.30      23950.24    7983.413333              4.0
11748         1.0         NaN  TXN-0001-0000-01-TXN1000666         1.0            31%  2021.0  ...             NaN             NaN              944

## Q139 - Check that referenced keys exist before attempting to delete related records

In [141]:
customerids = df["customer_id"].dropna().unique()
records_to_remove = df[
    df["customer_id"].isin(customerids)
]
print("Matching records found:", len(records_to_remove))

Matching records found: 12776


## Q140 - Create isolated copies of data for read-only analysis

In [142]:
readonlydata = df.copy()
customer_summary = readonlydata.groupby(
    "customer_id"
).agg(
    total_amount=("Credit_Amount", "sum"),
    transaction_count=("transaction_id", "count")
).reset_index()
print(customer_summary)

     customer_id  total_amount  transaction_count
0            1.0      94428.30                 25
1           10.0      15454.30                  3
2          100.0       5891.89                  2
3         1000.0      50201.61                  6
4         1001.0      12626.40                  2
...          ...           ...                ...
4313       992.0      21199.22                  5
4314       993.0       -102.51                  1
4315       994.0      18288.15                  4
4316       997.0      10000.00                  1
4317       998.0       1806.33                  1

[4318 rows x 3 columns]


## Q141 - Use parameterized values instead of string concatenation for dynamic filters

In [143]:
payment_status = "Success"
result = df[
    df["payment_status"] == payment_status
]
print(result)

      customer_id  account_id transaction_id  product_id product_margin  year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
13002      1001.0         NaN        TXN9999         NaN            NaN   NaN  ...     Credit_card          PayPal               12626.4       12626.4         6313.2              1.0

[1 rows x 50 columns]


## Q142 - Plan column structure changes before actively working with data

In [144]:
required_columns = [
    "customer_id",
    "account_id",
    "transaction_id",
    "txn_date",
    "txn_type",
    "payment_status",
    "payment_method",
    "Credit_Amount"
]
data = df[required_columns]
print(data.head())

      customer_id  account_id               transaction_id   txn_date    txn_type payment_status payment_method  Credit_Amount
671           1.0         NaN   TXN-0001-0000-01-TXN206851 2021-05-12  Withdrawal        pending    Credit Card       10000.00
8090          1.0         NaN   TXN-0001-0000-01-TXN627276 2021-10-22      Refund            NaN    Credit Card       10000.00
7359          1.0         NaN   TXN-0001-0000-01-TXN809570 2021-12-15    Purchase        success    Credit Card        3950.24
11748         1.0         NaN  TXN-0001-0000-01-TXN1000666 2021-12-15    Purchase        success    Credit Card        3950.24
2240          1.0         NaN   TXN-0001-0000-01-TXN322086 2022-03-01         NaN       refunded    Credit Card        2231.78


## Q143 - Respect auto-generated identifier columns during inserts

In [145]:
new_record = {
    "customer_id": "C1001",
    "account_id": "A1001",
    "txn_date": "2026-10-07",
    "txn_type": "Deposit",
    "payment_status": "Success",
    "payment_method": "Credit Card",
    "Credit_Amount": 10000
}
new_data = pd.DataFrame([new_record])
df = pd.concat(
    [df, new_data],
    ignore_index=True
)
print(df.tail())

      customer_id account_id              transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
12999         NaN     5513.0  TXN-1490-5513-20-TXN680820        20.0            43%  2026.0  ...             NaN             NaN                   NaN           NaN            NaN              NaN
13000         NaN     6778.0  TXN-0773-6778-22-TXN734409        22.0            19%  2025.0  ...             NaN             NaN                   NaN           NaN            NaN              NaN
13001         NaN     1287.0  TXN-3157-1287-20-TXN232264        20.0            22%  2022.0  ...             NaN             NaN                   NaN           NaN            NaN              NaN
13002         NaN     5071.0  TXN-3601-5071-18-TXN596972        18.0            72%  2024.0  ...             NaN             NaN                   NaN           NaN            NaN              NaN
13003       C10

## Q144 - Use filtering to exclude records instead of permanently deleting them

In [146]:
result = df[
    df["payment_status"] != "Failed"
]
print(result)

      customer_id account_id               transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
0             1.0        NaN   TXN-0001-0000-01-TXN206851         1.0            40%  2021.0  ...             NaN             NaN               94428.3      10000.00   10000.000000              1.0
1             1.0        NaN   TXN-0001-0000-01-TXN627276         1.0            99%  2021.0  ...             NaN             NaN               94428.3      20000.00   10000.000000              1.0
2             1.0        NaN   TXN-0001-0000-01-TXN809570         1.0            37%  2021.0  ...             NaN             NaN               94428.3      23950.24    7983.413333              4.0
3             1.0        NaN  TXN-0001-0000-01-TXN1000666         1.0            31%  2021.0  ...             NaN             NaN               94428.3      27900.48    5966.826667              4.0
4         

## Q145 - Group related operations in a single pipeline or chain

In [147]:
result = (
    df
    .assign(
        Credit_Amount=pd.to_numeric(
            df["Credit_Amount"],
            errors="coerce"
        )
    )
    .query("Credit_Amount > 10000")
    .groupby("customer_id")["Credit_Amount"]
    .sum()
    .reset_index()
)
print(result)

    customer_id  Credit_Amount
0        1000.0       36065.07
1        1007.0       10045.64
2         102.0       10501.83
3        1025.0       12648.95
4        1031.0       10263.46
..          ...            ...
271       901.0       11223.25
272       902.0       10221.55
273        94.0       12325.43
274       960.0       10230.88
275       970.0       11795.33

[276 rows x 2 columns]


## Q146 - Handle insert/update conflicts by identifying which records matched and which did not

In [148]:
new_data = pd.DataFrame({
    "customer_id": ["C001", "C999"],
    "Credit_Amount": [5000, 7000]
})
check = new_data.merge(
    df[["customer_id"]],
    on="customer_id",
    how="left",
    indicator=True
)
matched = check[check["_merge"] == "both"]
not_matched = check[check["_merge"] == "left_only"]
print("Matched records:")
print(matched)
print("Records not found:")
print(not_matched)

Matched records:
Empty DataFrame
Columns: [customer_id, Credit_Amount, _merge]
Index: []
Records not found:
  customer_id  Credit_Amount     _merge
0        C001           5000  left_only
1        C999           7000  left_only


## Q147 - Store date columns as proper date types instead of keeping them as text

In [149]:
df["txn_date"] = pd.to_datetime(
    df["txn_date"],
    errors="coerce"
)
print(df["txn_date"].dtype)

datetime64[us]


## Q148 - Always specify column names explicitly when creating new data structures

In [150]:
customer_summary = pd.DataFrame(
    df.groupby("customer_id")["Credit_Amount"]
    .sum()
    .reset_index()
)
customer_summary.columns = [
    "customer_id",
    "total_credit_amount"
]
print(customer_summary)

     customer_id  total_credit_amount
0            1.0             94428.30
1           10.0             15454.30
2          100.0              5891.89
3         1000.0             50201.61
4         1001.0             12626.40
...          ...                  ...
4314       993.0              -102.51
4315       994.0             18288.15
4316       997.0             10000.00
4317       998.0              1806.33
4318       C1001             10000.00

[4319 rows x 2 columns]


## Q149 - Apply soft-delete filters when querying data that has logical deletion markers

In [151]:
result = df[
    df["payment_status"] != "Refunded"
]
print(result)

      customer_id account_id               transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
0             1.0        NaN   TXN-0001-0000-01-TXN206851         1.0            40%  2021.0  ...             NaN             NaN               94428.3      10000.00   10000.000000              1.0
1             1.0        NaN   TXN-0001-0000-01-TXN627276         1.0            99%  2021.0  ...             NaN             NaN               94428.3      20000.00   10000.000000              1.0
2             1.0        NaN   TXN-0001-0000-01-TXN809570         1.0            37%  2021.0  ...             NaN             NaN               94428.3      23950.24    7983.413333              4.0
3             1.0        NaN  TXN-0001-0000-01-TXN1000666         1.0            31%  2021.0  ...             NaN             NaN               94428.3      27900.48    5966.826667              4.0
4         

## Q150 - Verify filters by previewing the filtered subset before applying updates

In [152]:
filtered_data = df[
    df["payment_status"] == "Success"
]
print(filtered_data.head())
df.loc[
    filtered_data.index,
    "payment_method"
] = df.loc[
    filtered_data.index,
    "payment_method"
].str.strip()
print(df.loc[filtered_data.index].head())

      customer_id account_id transaction_id  product_id product_margin  year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
37         1001.0        NaN        TXN9999         NaN            NaN   NaN  ...     Credit_card          PayPal               12626.4       12626.4         6313.2              1.0
13003       C1001      A1001            NaN         NaN            NaN   NaN  ...             NaN             NaN                   NaN           NaN            NaN              NaN

[2 rows x 50 columns]
      customer_id account_id transaction_id  product_id product_margin  year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
37         1001.0        NaN        TXN9999         NaN            NaN   NaN  ...     Credit_card          PayPal               12626.4       12626.4         6313.2              1.0
13003       C1001      A1001            NaN         NaN            

## Q151 - Avoid redundant validation checks already enforced by data types

In [153]:
df["Credit_Amount"] = pd.to_numeric(
    df["Credit_Amount"],
    errors="coerce"
)
result = df[
    df["Credit_Amount"] > 10000
]
print(result)

      customer_id account_id              transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
32         1000.0      362.0  TXN-1000-0362-16-TXN469391        16.0            80%  2024.0  ...             NaN             NaN              50201.61      14374.71    4791.570000              2.0
33         1000.0     5848.0  TXN-1000-5848-23-TXN796992        23.0            13%  2025.0  ...             NaN             NaN              50201.61      38302.01   12394.513333              1.0
53         1007.0     7272.0  TXN-1007-7272-20-TXN519532        20.0            33%  2021.0  ...             NaN             NaN              23274.89      10045.64   10045.640000              1.0
89          102.0     7757.0  TXN-0102-7757-24-TXN854942        24.0            NaN  2022.0  ...             NaN             NaN              24363.66      11333.20    5666.600000              1.0
100        1025

## Q152 - Base optimization decisions on current dataset size and memory usage

In [154]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("Memory usage in bytes:", df.memory_usage(deep=True).sum())
print(df.info())

Number of rows: 13004
Number of columns: 50
Memory usage in bytes: 21917207
<class 'pandas.DataFrame'>
RangeIndex: 13004 entries, 0 to 13003
Data columns (total 50 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   customer_id                   12777 non-null  str           
 1   account_id                    12701 non-null  object        
 2   transaction_id                12766 non-null  str           
 3   product_id                    12772 non-null  float64       
 4   product_margin                12742 non-null  str           
 5   year                          12745 non-null  float64       
 6   month                         12759 non-null  float64       
 7   region                        12735 non-null  str           
 8   customer_segment              12772 non-null  str           
 9   kyc_status                    12767 non-null  str           
 10  txn_id_external              

## Q153 - Plan large structural changes carefully and test on a sample first

In [155]:
sample_data = df.head(100).copy()
sample_data["Credit_Amount"] = pd.to_numeric(
    sample_data["Credit_Amount"],
    errors="coerce"
)
sample_data["txn_date"] = pd.to_datetime(
    sample_data["txn_date"],
    errors="coerce"
)
print(sample_data.head())

  customer_id account_id               transaction_id  product_id product_margin    year  ...  Payment_Method Payment_gateway customer_total_amount running_total moving_average transaction_rank
0         1.0        NaN   TXN-0001-0000-01-TXN206851         1.0            40%  2021.0  ...             NaN             NaN               94428.3      10000.00   10000.000000              1.0
1         1.0        NaN   TXN-0001-0000-01-TXN627276         1.0            99%  2021.0  ...             NaN             NaN               94428.3      20000.00   10000.000000              1.0
2         1.0        NaN   TXN-0001-0000-01-TXN809570         1.0            37%  2021.0  ...             NaN             NaN               94428.3      23950.24    7983.413333              4.0
3         1.0        NaN  TXN-0001-0000-01-TXN1000666         1.0            31%  2021.0  ...             NaN             NaN               94428.3      27900.48    5966.826667              4.0
4         1.0        NaN   TXN

## Q154 - Review any dependent processes or scripts before modifying data structure

In [156]:
required_columns = [
    "customer_id",
    "account_id",
    "transaction_id",
    "txn_date",
    "txn_type",
    "payment_status",
    "payment_method",
    "Credit_Amount"
]
missing_columns = df.columns[
    ~df.columns.isin(required_columns)
]
print("Columns available in dataset:")
print(df.columns.tolist())
print("\nColumns that are not part of the planned structure:")
print(missing_columns.tolist())

Columns available in dataset:
['customer_id', 'account_id', 'transaction_id', 'product_id', 'product_margin', 'year', 'month', 'region', 'customer_segment', 'kyc_status', 'txn_id_external', 'txn_type', 'txn_mode', 'payment_method', 'payment_gateway', 'payment_status', 'device_type', 'app_version', 'offer_id', 'offer_category', 'offer_applied_flag', 'reward_points_earned', 'offer_effectiveness', 'account_balance', 'Fixed_Deposite', 'Credit_Amount', 'txn_date', 'fraud_risk_level', 'aml_flag', 'suspicious_txn_flag', 'clv', 'avg_transaction_value', 'txn_frequency_30d', 'customer_activity_status', 'most_common_txn_time_of_day', 'averag_high_txn_weekend_flag', 'averag_high_txn_day_of_week', 'txn_velocity_1hr', 'device_change_flag', 'location_change_flag', 'failed_txn_count_24hr', 'balance_category', 'value_group', 'Payment_status', 'Payment_Method', 'Payment_gateway', 'customer_total_amount', 'running_total', 'moving_average', 'transaction_rank']

Columns that are not part of the planned str

# Section J - Second Dataset (Loan Records) & Cross-Dataset Integrity (J1-J10)

Review fixes covered here:
- **J1** load + profile Loan_Records
- **J2** parse `loan_offer_valid_from` / `loan_offer_valid_to` (format-specific)
- **J3** detect day/month swaps in offer dates
- **J4** validate the offer-date range
- **J5** handle block-missing loan rows
- **J6** cardinality-controlled join (`validate="m:1"`)
- **J7** quantify orphan keys
- **J8** composite-key validity on the loan side
- **J9** align key/flag types before joining
- **J10** handle interest-rate units

In [ ]:
LOAN = Path.cwd() / "Loan Records 1.xlsx"
if not LOAN.exists():
    LOAN = next(Path.cwd().rglob("Loan Records 1.xlsx"))
loan_raw = pd.read_excel(LOAN)
loan = loan_raw.copy()
print("LOAN =", LOAN)
print("shape:", loan.shape)
print("\ndtypes:")
print(loan.dtypes.to_string())
print("\nnulls per column:")
print(loan.isna().sum().to_string())
print("\nhead:")
print(loan.head(3).to_string())

In [ ]:
# J2: parse both offer-date columns with the same format-aware parser used
# for txn_date (handles ISO, 'January8-January-2021', '22/01/2021', '212021').
loan["offer_from"] = loan["loan_offer_valid_from"].apply(to_date)
loan["offer_to"] = loan["loan_offer_valid_to"].apply(to_date)
print("parse failures  from:", int(loan["offer_from"].isna().sum()),
      "| to:", int(loan["offer_to"].isna().sum()))
print("\nsample conversions:")
print(loan[["loan_offer_valid_from", "offer_from"]].drop_duplicates().head(8).to_string())

In [ ]:
# J3: day/month swap detection - compare day-first vs month-first parses
# of the ambiguous numeric dates (same helper as txn_date, C5/C6).
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _mf = loan["loan_offer_valid_from"].apply(
        lambda v: pd.to_datetime(v, errors="coerce") if isinstance(v, str) else pd.NaT)
_amb = loan["loan_offer_valid_from"].astype(str).str.match(r"^\d{1,2}/\d{1,2}/\d{2,4}$")
_sw = int((_amb & loan["offer_from"].notna() & _mf.notna() & (loan["offer_from"] != _mf)).sum())
print("ambiguous numeric offer dates:", int(_amb.sum()), "| day/month swap candidates:", _sw)

In [ ]:
# J4: offer-date range validation
print("offer_from range:", loan["offer_from"].min(), "->", loan["offer_from"].max())
print("offer_to   range:", loan["offer_to"].min(), "->", loan["offer_to"].max())
_bad = int((loan["offer_to"] < loan["offer_from"]).sum())
print("rows where offer_to < offer_from:", _bad)

In [ ]:
# J5: block-missing rows - rows where every loan attribute is null
_all_null = loan[loan[["min_interest_rate", "loan_type", "collateral_flag",
                       "interest_type", "last_loan_status", "Othr_loans_emi_status",
                       "last_loan_payment_mode", "loan_offer_valid_from",
                       "loan_offer_valid_to", "fraud_flag"]].isna().all(axis=1)]
print("block-missing loan rows:", len(_all_null))
loan = loan[~loan.index.isin(_all_null.index)].copy()
print("shape after dropping block-missing rows:", loan.shape)

In [ ]:
# J6: cardinality-controlled join. Loan_Records has duplicate customer_ids,
# so aggregate to ONE row per customer (latest offer) and use validate="m:1".
print("loan rows:", len(loan), "| unique customers:", loan["customer_id"].nunique(),
      "| duplicate customer_ids:", int(loan["customer_id"].duplicated().sum()))
loan_latest = (loan.sort_values("offer_from")
                  .drop_duplicates("customer_id", keep="last"))
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")
joined = df.merge(loan_latest, on="customer_id", how="left", validate="m:1")
print("bank rows before:", len(df), "| after join:", len(joined),
      "(equal => no row explosion)")

In [ ]:
# J7: orphan keys - bank customers with no loan record and vice versa
_chk = df.merge(loan_latest[["customer_id"]], on="customer_id", how="left", indicator=True)
_orph = int((_chk["_merge"] == "left_only").sum())
print("bank transactions with no loan record:", _orph, f"({_orph/len(df):.1%})")
_bank_ids = set(df["customer_id"].dropna())
print("loan customers with no bank activity:",
      int((~loan_latest["customer_id"].isin(_bank_ids)).sum()))

In [ ]:
# J8: composite-key validity on the loan side
_dup = loan.duplicated(subset=["customer_id", "account_id"], keep=False).sum()
print("loan rows sharing a (customer_id, account_id) pair:", int(_dup))
print("=> customer_id alone is NOT unique; the m:1 aggregation in J6 is required.")

In [ ]:
# J9: key/flag type alignment
loan["customer_id"] = pd.to_numeric(loan["customer_id"], errors="coerce").astype("Int64")
loan["fraud_flag"] = loan["fraud_flag"].apply(unify_flag)          # 0/1 + one 'Yes'
loan["collateral_flag"] = pd.to_numeric(loan["collateral_flag"], errors="coerce").astype("Int64")
print(loan[["customer_id", "fraud_flag", "collateral_flag"]].dtypes.to_string())
print("fraud_flag:", loan["fraud_flag"].value_counts(dropna=False).to_dict())

In [ ]:
# J10: interest-rate units - values 6-18 are PERCENT, store a fraction column too
_rate = pd.to_numeric(loan["min_interest_rate"], errors="coerce")
print("min/max rate:", _rate.min(), _rate.max())
loan["min_interest_rate_pct"] = _rate
loan["min_interest_rate_frac"] = _rate / 100
print(loan[["min_interest_rate_pct", "min_interest_rate_frac"]].head(3).to_string())

# Section J - Second Dataset (Loan Records) & Cross-Dataset Integrity (J1-J10)

Review fixes covered here:
- **J1** load + profile Loan_Records
- **J2** parse `loan_offer_valid_from` / `loan_offer_valid_to` (format-specific)
- **J3** detect day/month swaps in offer dates
- **J4** validate the offer-date range
- **J5** handle block-missing loan rows
- **J6** cardinality-controlled join (`validate="m:1"`)
- **J7** quantify orphan keys
- **J8** composite-key validity on the loan side
- **J9** align key/flag types before joining
- **J10** handle interest-rate units

In [ ]:
LOAN = Path.cwd() / "Loan Records 1.xlsx"
if not LOAN.exists():
    LOAN = next(Path.cwd().rglob("Loan Records 1.xlsx"))
loan_raw = pd.read_excel(LOAN)
loan = loan_raw.copy()
print("LOAN =", LOAN)
print("shape:", loan.shape)
print("\ndtypes:")
print(loan.dtypes.to_string())
print("\nnulls per column:")
print(loan.isna().sum().to_string())
print("\nhead:")
print(loan.head(3).to_string())

In [ ]:
# J2: parse both offer-date columns with the same format-aware parser used
# for txn_date (handles ISO, 'January8-January-2021', '22/01/2021', '212021').
loan["offer_from"] = loan["loan_offer_valid_from"].apply(to_date)
loan["offer_to"] = loan["loan_offer_valid_to"].apply(to_date)
print("parse failures  from:", int(loan["offer_from"].isna().sum()),
      "| to:", int(loan["offer_to"].isna().sum()))
print("\nsample conversions:")
print(loan[["loan_offer_valid_from", "offer_from"]].drop_duplicates().head(8).to_string())

In [ ]:
# J3: day/month swap detection - compare day-first vs month-first parses
# of the ambiguous numeric dates (same helper as txn_date, C5/C6).
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _mf = loan["loan_offer_valid_from"].apply(
        lambda v: pd.to_datetime(v, errors="coerce") if isinstance(v, str) else pd.NaT)
_amb = loan["loan_offer_valid_from"].astype(str).str.match(r"^\d{1,2}/\d{1,2}/\d{2,4}$")
_sw = int((_amb & loan["offer_from"].notna() & _mf.notna() & (loan["offer_from"] != _mf)).sum())
print("ambiguous numeric offer dates:", int(_amb.sum()), "| day/month swap candidates:", _sw)

In [ ]:
# J4: offer-date range validation
print("offer_from range:", loan["offer_from"].min(), "->", loan["offer_from"].max())
print("offer_to   range:", loan["offer_to"].min(), "->", loan["offer_to"].max())
_bad = int((loan["offer_to"] < loan["offer_from"]).sum())
print("rows where offer_to < offer_from:", _bad)

In [ ]:
# J5: block-missing rows - rows where every loan attribute is null
_all_null = loan[loan[["min_interest_rate", "loan_type", "collateral_flag",
                       "interest_type", "last_loan_status", "Othr_loans_emi_status",
                       "last_loan_payment_mode", "loan_offer_valid_from",
                       "loan_offer_valid_to", "fraud_flag"]].isna().all(axis=1)]
print("block-missing loan rows:", len(_all_null))
loan = loan[~loan.index.isin(_all_null.index)].copy()
print("shape after dropping block-missing rows:", loan.shape)

In [ ]:
# J6: cardinality-controlled join. Loan_Records has duplicate customer_ids,
# so aggregate to ONE row per customer (latest offer) and use validate="m:1".
print("loan rows:", len(loan), "| unique customers:", loan["customer_id"].nunique(),
      "| duplicate customer_ids:", int(loan["customer_id"].duplicated().sum()))
loan_latest = (loan.sort_values("offer_from")
                  .drop_duplicates("customer_id", keep="last"))
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")
joined = df.merge(loan_latest, on="customer_id", how="left", validate="m:1")
print("bank rows before:", len(df), "| after join:", len(joined),
      "(equal => no row explosion)")

In [ ]:
# J7: orphan keys - bank customers with no loan record and vice versa
_chk = df.merge(loan_latest[["customer_id"]], on="customer_id", how="left", indicator=True)
_orph = int((_chk["_merge"] == "left_only").sum())
print("bank transactions with no loan record:", _orph, f"({_orph/len(df):.1%})")
_bank_ids = set(df["customer_id"].dropna())
print("loan customers with no bank activity:",
      int((~loan_latest["customer_id"].isin(_bank_ids)).sum()))

In [ ]:
# J8: composite-key validity on the loan side
_dup = loan.duplicated(subset=["customer_id", "account_id"], keep=False).sum()
print("loan rows sharing a (customer_id, account_id) pair:", int(_dup))
print("=> customer_id alone is NOT unique; the m:1 aggregation in J6 is required.")

In [ ]:
# J9: key/flag type alignment
loan["customer_id"] = pd.to_numeric(loan["customer_id"], errors="coerce").astype("Int64")
loan["fraud_flag"] = loan["fraud_flag"].apply(unify_flag)          # 0/1 + one 'Yes'
loan["collateral_flag"] = pd.to_numeric(loan["collateral_flag"], errors="coerce").astype("Int64")
print(loan[["customer_id", "fraud_flag", "collateral_flag"]].dtypes.to_string())
print("fraud_flag:", loan["fraud_flag"].value_counts(dropna=False).to_dict())

In [ ]:
# J10: interest-rate units - values 6-18 are PERCENT, store a fraction column too
_rate = pd.to_numeric(loan["min_interest_rate"], errors="coerce")
print("min/max rate:", _rate.min(), _rate.max())
loan["min_interest_rate_pct"] = _rate
loan["min_interest_rate_frac"] = _rate / 100
print(loan[["min_interest_rate_pct", "min_interest_rate_frac"]].head(3).to_string())